In [ ]:
import pandas as pd
from functools import lru_cache
import ast
import re

import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)
import sys
sys.path.append("/Users/tahreemyasir/Documents/prelims/DT_hint-1")

from dt_code.GPT.csv_baseline_1 import convert_rule_to_short_name
from dt_code.KG.KG_traversal import load_nodes_and_parents


# Loading data into separate df's 

In [ ]:
df_1 = pd.read_csv('/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/csv/llama3/llama_baseline_1.csv')
df_2 = pd.read_csv('/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/csv/llama3/llama_baseline_2.csv')
df_3 = pd.read_csv('/Users/tahreemyasir/Documents/prelims/DT_hint-1/Data/csv/llama3/llama_ours.csv')

# Convert df_1['id'] from string format like '[450]' to integer 450
# This ensures all DataFrames have the same dtype for the 'id' column
df_1['id'] = df_1['id'].astype(str).str.strip('[]').astype(int)

# df_1.isnull().sum()   
print("original number of rows baseline 1: ", len(df_1))
print("original number of rows baseline 2: ", len(df_2))
print("original number of rows baseline 3: ", len(df_3))

print("unique ids baseline 1: ", df_1['id'].nunique())
print("unique ids baseline 2: ", df_2['id'].nunique())
print("unique ids baseline 3: ", df_3['id'].nunique())

print("columns baseline 1: ", df_1.shape[1])
print("columns baseline 2: ", df_2.shape[1])
print("columns baseline 3: ", df_3.shape[1])

# Convert all id columns to the same type
df_1['id'] = df_1['id'].astype(str).str.strip('[]').astype(int)
df_2['id'] = df_2['id'].astype(str).str.strip('[]').astype(int)
df_3['id'] = df_3['id'].astype(str).str.strip('[]').astype(int)

# Now merge
df = (
    df_1
    .merge(df_2, on="id", how="inner")
    .merge(df_3, on="id", how="inner")
)

print("columns merged: ", df.shape[1])
print("rows merged: ", df.shape[0])
print("unique ids merged: ", df['id'].nunique())
df.isnull().sum()


#apply convert_rule_to_short_name(rule_name) to student_rule, student_update_rule, judge_student_update_rule, ours_student_update_rule from gpt/csv_baseline_1.py
from dt_code.GPT.csv_baseline_1 import convert_rule_to_short_name
# remove space from KG_rule, student_rule, student_update_rule, judge_student_update_rule, ours_student_update_rule
df['KG_rule'] = df['KG_rule'].str.replace(' ', '')
df['student_rule'] = df['student_rule'].str.replace(' ', '')
df['student_update_rule'] = df['student_update_rule'].str.replace(' ', '')
df['judge_student_update_rule'] = df['judge_student_update_rule'].str.replace(' ', '')
df['ours_student_update_rule'] = df['ours_student_update_rule'].str.replace(' ', '')

for col in [
    'KG_rule',
    'student_rule', 
    'student_update_rule',
    'judge_student_update_rule',
    'ours_student_update_rule'
]:
    if col in df.columns:
        df[col] = df[col].astype(str).apply(convert_rule_to_short_name)



# KG connection establish 

In [ ]:
# Create Neo4j connection
from dt_code.KG.KG_traversal import Neo4jConnection
from dt_code.utils.env_loader import load_env
import os

load_env()
URI = os.environ.get('NEO4J_URI')
AUTH = (os.environ.get('NEO4J_USERNAME'), os.environ.get('NEO4J_PASSWORD'))
conn = Neo4jConnection(URI, AUTH[0], AUTH[1])
print("Neo4j connection established")


# Creating comparison dictionaries 

In [ ]:
# Normalize expression to handle equivalent forms like "(A+B)" and "A+B"
def normalize_expression(expr):
    """
    Normalize expression by removing spaces and handling outer parentheses.
    This makes "(A+B)" and "A+B" equivalent for comparison purposes.
    
    Handles edge cases:
    - None, NaN, empty strings
    - Multiple nested outer parentheses: ((A+B)) -> A+B
    - Preserves necessary parentheses: (A+B)*C stays as (A+B)*C
    """
    # Handle None, NaN, and empty values
    if expr is None or pd.isna(expr):
        return ''
    
    expr_str = str(expr).strip()
    
    # Return empty string if nothing left after stripping
    if not expr_str:
        return ''
    
    # Remove all spaces
    expr_str = expr_str.replace(' ', '')
    
    # Remove outer parentheses if the entire expression is wrapped
    # Only remove if the opening '(' at position 0 matches the closing ')' at the end
    # This handles cases like ((A+B)) -> A+B, but preserves (A+B)*C
    while len(expr_str) > 0 and expr_str.startswith('(') and expr_str.endswith(')'):
        # Check if the first '(' matches the last ')'
        # Count parentheses from the start - when we hit 0, that's where the matching ')' is
        paren_count = 0
        matching_pos = -1
        for i, char in enumerate(expr_str):
            if char == '(':
                paren_count += 1
            elif char == ')':
                paren_count -= 1
                if paren_count == 0:
                    matching_pos = i
                    break
        
        # Only remove if the matching ')' is at the very end
        if matching_pos == len(expr_str) - 1 and len(expr_str) > 2:
            expr_str = expr_str[1:-1]  # Remove outer parentheses
        else:
            break  # Don't remove if parentheses don't wrap the entire expression
    
    return expr_str

# Helper function to normalize a list of parent expressions
def normalize_parent_list(parents):
    """Normalize a list of parent expressions, filtering out None/empty values"""
    if not parents:
        return []
    normalized = []
    for p in parents:
        norm_p = normalize_expression(p)
        if norm_p:  # Only add non-empty normalized expressions
            normalized.append(norm_p)
    return normalized

# Create two dictionaries: kg_optimal_dict and kg_alternative_dict
def create_kg_dictionaries(row):
    """Create kg_optimal_dict and kg_alternative_dict for each row"""
    kg_optimal_dict = {}
    kg_alternative_dict = {}
    
    try:
        # Get problem info
        problem_number = str(row['problem_number'])
        kg_step = row.get('KG_step', '')
        kg_rule = convert_rule_to_short_name(str(row.get('KG_rule', '')))
        known_expressions = row.get('known_expressions', [])
        
        # Parse known_expressions if string
        if isinstance(known_expressions, str):
            known_expressions = ast.literal_eval(known_expressions) if known_expressions else []
        
        # Normalize known expressions for robust comparison
        known_expressions_normalized = {normalize_expression(expr) for expr in known_expressions}
        
        # Load all derivations from KG
        derivations = load_nodes_and_parents(conn, problem_number)
        
        # Process each expression in derivations
        for expr, derivation_routes in derivations.items():
            # Normalize expression for consistent comparison
            expr_normalized = normalize_expression(expr)
            
            # Skip known expressions (givens) - using normalized forms
            if expr_normalized in known_expressions_normalized:
                continue
            
            # Normalize kg_step for consistent comparison
            kg_step_normalized = normalize_expression(kg_step)
            
            # Build entries for this expression
            entries = []
            for route in derivation_routes:
                if route:
                    rule = convert_rule_to_short_name(route[-1])  # Last element is rule
                    parents = route[:-1]  # All except last are parents
                    # Normalize parent expressions using helper function
                    parents_normalized = normalize_parent_list(parents)
                    entries.append({"rule": rule, "parents": parents_normalized})
            
            if not entries:
                continue
            
            # Check if this is the optimal step (using normalized forms)
            if expr_normalized == kg_step_normalized:
                # Add to optimal dict (only entries matching KG_rule)
                optimal_entries = [e for e in entries if e["rule"] == kg_rule]
                if optimal_entries:
                    kg_optimal_dict[expr_normalized] = optimal_entries
                # Add remaining entries to alternative dict
                alternative_entries = [e for e in entries if e["rule"] != kg_rule]
                if alternative_entries:
                    kg_alternative_dict[expr_normalized] = alternative_entries
            else:
                # Add all entries to alternative dict
                kg_alternative_dict[expr_normalized] = entries
        
        return kg_optimal_dict, kg_alternative_dict
    
    except Exception as e:
        print(f"Error processing row {row.get('id', 'unknown')}: {e}")
        return {}, {}

# Apply function to create dictionaries
print("Creating kg_optimal_dict and kg_alternative_dict...")
results = df.apply(create_kg_dictionaries, axis=1)
df['kg_optimal_dict'] = [r[0] for r in results]
df['kg_alternative_dict'] = [r[1] for r in results]

print(f"✅ Created dictionaries for {len(df)} rows")
print(f"Sample kg_optimal_dict: {df['kg_optimal_dict'].iloc[0]}")
print(f"Sample kg_alternative_dict keys count: {len(df['kg_alternative_dict'].iloc[0])}")


# Optimal Comparison 

In [ ]:
# Compare student step, rule, and parents with kg_optimal_dict
def check_student_optimal_match(row):
    """Check if student step, rule, and parents match kg_optimal_dict"""
    try:
        student_step = row.get('student_step', '')
        student_rule = convert_rule_to_short_name(str(row.get('student_rule', '')))
        student_parents = row.get('student_parent_statements', [])
        
        # Parse student_parents if string
        if isinstance(student_parents, str):
            student_parents = ast.literal_eval(student_parents) if student_parents else []
        if student_parents is None:
            student_parents = []
        
        # Normalize student step and parents using helper function
        student_step_normalized = normalize_expression(student_step)
        student_parents_normalized = normalize_parent_list(student_parents)
        
        # Convert to set for order-independent comparison
        student_parents_set = set(student_parents_normalized)
        
        # Get kg_optimal_dict
        kg_optimal_dict = row.get('kg_optimal_dict', {})
        
        # Check if normalized student_step is in kg_optimal_dict
        if student_step_normalized not in kg_optimal_dict:
            return 0
        
        # Check all entries for this step
        entries = kg_optimal_dict[student_step_normalized]
        for entry in entries:
            entry_rule = entry.get('rule', '')
            entry_parents = entry.get('parents', [])
            entry_parents_set = set(entry_parents) if entry_parents else set()
            
            # Check if rule and parents match
            if entry_rule == student_rule and entry_parents_set == student_parents_set:
                return 1
        
        return 0
    
    except Exception as e:
        print(f"Error checking row {row.get('id', 'unknown')}: {e}")
        return 0

# Create student_count column
print("Comparing student predictions with kg_optimal_dict...")
df['student_count'] = df.apply(check_student_optimal_match, axis=1)

print(f"✅ Created student_count column")
print(f"Rows with match (student_count=1): {df['student_count'].sum()} ({df['student_count'].mean()*100:.1f}%)")
print(f"Rows without match (student_count=0): {(df['student_count']==0).sum()} ({(df['student_count']==0).mean()*100:.1f}%)")


# Filter Non_optimal 

In [ ]:
# Create new dataframe with rows where student_count = 0
df_non_optimal = df[df['student_count'] == 0].copy()

print(f"✅ Created df_non_optimal")
print(f"Original df rows: {len(df)}")
print(f"df_non_optimal rows: {len(df_non_optimal)} ({len(df_non_optimal)/len(df)*100:.1f}%)")
print(f"Rows with student_count=1: {len(df[df['student_count'] == 1])}")
print(f"Rows with student_count=0: {len(df_non_optimal)}")


# Non optimal assessment 

In [ ]:
# Compare student step, rule, and parents with kg_alternative_dict for df_non_optimal
def check_student_alternative_match(row):
    """Check if student step, rule, and parents match kg_alternative_dict"""
    result = {
        'expression_match': False,
        'rule_match': False,
        'parents_match': False,
        'is_correct': False
    }
    
    try:
        student_step = row.get('student_step', '')
        student_rule = convert_rule_to_short_name(str(row.get('student_rule', '')))
        student_parents = row.get('student_parent_statements', [])
        
        # Parse student_parents if string
        if isinstance(student_parents, str):
            student_parents = ast.literal_eval(student_parents) if student_parents else []
        if student_parents is None:
            student_parents = []
        
        # Normalize student step and parents using helper function
        student_step_normalized = normalize_expression(student_step)
        student_parents_normalized = normalize_parent_list(student_parents)
        
        # Convert to set for order-independent comparison
        student_parents_set = set(student_parents_normalized)
        
        # Get kg_alternative_dict
        kg_alternative_dict = row.get('kg_alternative_dict', {})
        
        # Check if normalized student_step is in kg_alternative_dict
        if student_step_normalized not in kg_alternative_dict:
            return result
        
        result['expression_match'] = True
        
        # Check all entries for this step
        entries = kg_alternative_dict[student_step_normalized]
        for entry in entries:
            entry_rule = entry.get('rule', '')
            entry_parents = entry.get('parents', [])
            entry_parents_set = set(entry_parents) if entry_parents else set()
            
            # Check if rule matches
            if entry_rule == student_rule:
                result['rule_match'] = True
                
                # Check if parents match
                if entry_parents_set == student_parents_set:
                    result['parents_match'] = True
                    result['is_correct'] = True
                    return result
        
        return result
    
    except Exception as e:
        print(f"Error checking row {row.get('id', 'unknown')}: {e}")
        return result

# Apply comparison function to df_non_optimal
print("Comparing student predictions in df_non_optimal with kg_alternative_dict...")
df_non_optimal['alternative_match_eval'] = df_non_optimal.apply(check_student_alternative_match, axis=1)

# Extract individual components
df_non_optimal['alternative_expression_match'] = df_non_optimal['alternative_match_eval'].apply(lambda x: x.get('expression_match', False))
df_non_optimal['alternative_rule_match'] = df_non_optimal['alternative_match_eval'].apply(lambda x: x.get('rule_match', False))
df_non_optimal['alternative_parents_match'] = df_non_optimal['alternative_match_eval'].apply(lambda x: x.get('parents_match', False))
df_non_optimal['alternative_is_correct'] = df_non_optimal['alternative_match_eval'].apply(lambda x: x.get('is_correct', False))

# Create count_sub column: 
# 0 = no match, 1 = step match, 2 = step and rule match, 3 = all match (step + rule + parents)
def check_count_sub(row):
    """Check match level: 0=no match, 1=step, 2=step+rule, 3=step+rule+parents"""
    try:
        student_step = row.get('student_step', '')
        student_rule = convert_rule_to_short_name(str(row.get('student_rule', '')))
        student_parents = row.get('student_parent_statements', [])
        
        # Parse student_parents if string
        if isinstance(student_parents, str):
            student_parents = ast.literal_eval(student_parents) if student_parents else []
        if student_parents is None:
            student_parents = []
        
        # Normalize student step and parents using helper function
        student_step_normalized = normalize_expression(student_step)
        student_parents_normalized = normalize_parent_list(student_parents)
        
        # Convert to set for order-independent comparison
        student_parents_set = set(student_parents_normalized)
        
        # Get kg_alternative_dict
        kg_alternative_dict = row.get('kg_alternative_dict', {})
        
        # Check if normalized student_step is in kg_alternative_dict
        if student_step_normalized not in kg_alternative_dict:
            return 0  # No match
        
        # Step matches (value = 1)
        entries = kg_alternative_dict[student_step_normalized]
        
        # Check all entries for this step
        for entry in entries:
            entry_rule = entry.get('rule', '')
            entry_parents = entry.get('parents', [])
            entry_parents_set = set(entry_parents) if entry_parents else set()
            
            # Check if rule matches
            if entry_rule == student_rule:
                # Rule matches (value = 2)
                # Check if parents match
                if entry_parents_set == student_parents_set:
                    return 3  # All match (step + rule + parents)
                return 2  # Step and rule match
        
        return 1  # Only step matches
    
    except Exception as e:
        print(f"Error checking row {row.get('id', 'unknown')}: {e}")
        return 0

df_non_optimal['count_sub'] = df_non_optimal.apply(check_count_sub, axis=1)

# Print results
print(f"\n✅ Comparison complete for {len(df_non_optimal)} rows")
print(f"\nResults:")
print(f"  Expression matches: {df_non_optimal['alternative_expression_match'].sum()} ({df_non_optimal['alternative_expression_match'].mean()*100:.1f}%)")
print(f"  Rule matches (when expression matches): {df_non_optimal['alternative_rule_match'].sum()} ({df_non_optimal['alternative_rule_match'].mean()*100:.1f}%)")
print(f"  Parents matches (when rule matches): {df_non_optimal['alternative_parents_match'].sum()} ({df_non_optimal['alternative_parents_match'].mean()*100:.1f}%)")
print(f"  Fully correct (expression + rule + parents): {df_non_optimal['alternative_is_correct'].sum()} ({df_non_optimal['alternative_is_correct'].mean()*100:.1f}%)")
print(f"\n  count_sub breakdown:")
print(f"    count_sub=0 (no match): {(df_non_optimal['count_sub']==0).sum()} ({(df_non_optimal['count_sub']==0).mean()*100:.1f}%)")
print(f"    count_sub=1 (step match): {(df_non_optimal['count_sub']==1).sum()} ({(df_non_optimal['count_sub']==1).mean()*100:.1f}%)")
print(f"    count_sub=2 (step + rule match): {(df_non_optimal['count_sub']==2).sum()} ({(df_non_optimal['count_sub']==2).mean()*100:.1f}%)")
print(f"    count_sub=3 (all match): {(df_non_optimal['count_sub']==3).sum()} ({(df_non_optimal['count_sub']==3).mean()*100:.1f}%)")


# non optimal complete match after update 

In [ ]:
# Complete match comparison for df_non_optimal only
# This compares step + rule + parents (all three must match)
def check_update_optimal_match(row, step_col, rule_col, parents_col):
    """Check if update step, rule, and parents match kg_optimal_dict (complete match only)"""
    try:
        update_step = row.get(step_col, '')
        update_rule = convert_rule_to_short_name(str(row.get(rule_col, '')))
        update_parents = row.get(parents_col, [])
        
        # Handle missing values
        if pd.isna(update_step) or update_step == '':
            return 0
        if pd.isna(update_rule) or update_rule == '':
            return 0
        
        # Parse update_parents if string
        if isinstance(update_parents, str):
            update_parents = ast.literal_eval(update_parents) if update_parents else []
        if update_parents is None:
            update_parents = []
        
        # Normalize update step and parents using helper function
        update_step_normalized = normalize_expression(update_step)
        update_parents_normalized = normalize_parent_list(update_parents)
        
        # Convert to set for order-independent comparison
        update_parents_set = set(update_parents_normalized)
        
        # Get kg_optimal_dict
        kg_optimal_dict = row.get('kg_optimal_dict', {})
        
        # Check if normalized update_step is in kg_optimal_dict
        if update_step_normalized not in kg_optimal_dict:
            return 0
        
        # Check all entries for this step
        entries = kg_optimal_dict[update_step_normalized]
        for entry in entries:
            entry_rule = entry.get('rule', '')
            entry_parents = entry.get('parents', [])
            entry_parents_set = set(entry_parents) if entry_parents else set()
            
            # Check if rule and parents match (COMPLETE MATCH)
            if entry_rule == update_rule and entry_parents_set == update_parents_set:
                return 1
        
        return 0
    
    except Exception as e:
        return 0

print("=" * 80)
print("COMPLETE MATCH COMPARISON: Update predictions vs kg_optimal_dict (df_non_optimal only)")
print("=" * 80)
print(f"\nTotal rows in df_non_optimal: {len(df_non_optimal)}")
print("(Complete match = step + rule + parents all match the optimal KG)")
print("(These are rows where initial student prediction did NOT match optimal KG)")

# Check complete matches for updates in df_non_optimal
df_non_optimal['student_update_complete'] = df_non_optimal.apply(
    lambda row: check_update_optimal_match(row, 'student_update_step', 'student_update_rule', 'student_update_parent_statements'),
    axis=1
)

df_non_optimal['judge_student_update_complete'] = df_non_optimal.apply(
    lambda row: check_update_optimal_match(row, 'judge_student_update_step', 'judge_student_update_rule', 'judge_student_update_parent_statements'),
    axis=1
)

df_non_optimal['ours_student_update_complete'] = df_non_optimal.apply(
    lambda row: check_update_optimal_match(row, 'ours_student_update_step', 'ours_student_update_rule', 'ours_student_update_parent_statements'),
    axis=1
)

# Also check initial student complete matches in kg_alternative_dict (count_sub=3)
initial_complete = (df_non_optimal['count_sub']==3).sum()

# Create three separate count columns: 1 if complete match, 0 if not
df_non_optimal['count_student_update'] = df_non_optimal['student_update_complete'].astype(int)
df_non_optimal['count_judge_update'] = df_non_optimal['judge_student_update_complete'].astype(int)
df_non_optimal['count_ours_update'] = df_non_optimal['ours_student_update_complete'].astype(int)

# Create three label columns with binary labels: "Complete Match" (1) or "No Match" (0)
# Convert to int first to ensure we have 0/1 values, then map to labels
df_non_optimal['student_update_label'] = df_non_optimal['student_update_complete'].astype(int).map({1: 'Complete Match', 0: 'No Match'})
df_non_optimal['judge_student_update_label'] = df_non_optimal['judge_student_update_complete'].astype(int).map({1: 'Complete Match', 0: 'No Match'})
df_non_optimal['ours_student_update_label'] = df_non_optimal['ours_student_update_complete'].astype(int).map({1: 'Complete Match', 0: 'No Match'})

# Verify the columns were created
print(f"\n✅ Created label columns:")
print(f"  - student_update_label: {df_non_optimal['student_update_label'].value_counts().to_dict()}")
print(f"  - judge_student_update_label: {df_non_optimal['judge_student_update_label'].value_counts().to_dict()}")
print(f"  - ours_student_update_label: {df_non_optimal['ours_student_update_label'].value_counts().to_dict()}")

print("\n" + "=" * 80)
print("RESULTS: Complete Matches (step + rule + parents)")
print("=" * 80)

print(f"\nInitial student (complete match in alternative dict): {initial_complete} ({initial_complete/len(df_non_optimal)*100:.1f}%)")
print(f"Student update (complete match in optimal dict): {df_non_optimal['student_update_complete'].sum()} ({df_non_optimal['student_update_complete'].mean()*100:.1f}%)")
print(f"Judge student update (complete match in optimal dict): {df_non_optimal['judge_student_update_complete'].sum()} ({df_non_optimal['judge_student_update_complete'].mean()*100:.1f}%)")
print(f"Ours student update (complete match in optimal dict): {df_non_optimal['ours_student_update_complete'].sum()} ({df_non_optimal['ours_student_update_complete'].mean()*100:.1f}%)")






# Next step correctness optimal 

In [ ]:
# Complete matches of student vs assessment from each agent (next_step_correctness)
print("=" * 80)
print("Student Complete Matches vs Agent Assessments")
print("=" * 80)

if 'next_step_correctness_x' in df.columns and 'next_step_correctness_y' in df.columns and 'next_step_correctness_ours' in df.columns:
    # Filter to only rows where student has complete match (student_count=1)
    df_complete = df[df['student_count'] == 1].copy()
    
    print(f"\nTotal rows with student complete match: {len(df_complete)}")
    
    print("\nTeacher Assessment (next_step_correctness_x):")
    print(df_complete['next_step_correctness_x'].value_counts().to_frame('Count'))
    
    print("\nJudge Assessment (next_step_correctness_y):")
    print(df_complete['next_step_correctness_y'].value_counts().to_frame('Count'))
    
    print("\nOurs Assessment (next_step_correctness_ours):")
    print(df_complete['next_step_correctness_ours'].value_counts().to_frame('Count'))
    
    # Summary table
    summary_data = {
        'Assessment': ['Correct', 'Incorrect', 'Suboptimal'],
        'Teacher': [
            (df_complete['next_step_correctness_x'] == 'Correct').sum(),
            (df_complete['next_step_correctness_x'] == 'Incorrect').sum(),
            (df_complete['next_step_correctness_x'] == 'Suboptimal').sum()
        ],
        'Judge': [
            (df_complete['next_step_correctness_y'] == 'Correct').sum(),
            (df_complete['next_step_correctness_y'] == 'Incorrect').sum(),
            (df_complete['next_step_correctness_y'] == 'Suboptimal').sum()
        ],
        'Ours': [
            (df_complete['next_step_correctness_ours'] == 'Correct').sum(),
            (df_complete['next_step_correctness_ours'] == 'Incorrect').sum(),
            (df_complete['next_step_correctness_ours'] == 'Suboptimal').sum()
        ]
    }
    
    summary_df = pd.DataFrame(summary_data)
    print("\n" + "=" * 80)
    print("Summary Table: Student Complete Matches - Agent Assessments")
    print("=" * 80)
    print(summary_df.to_string(index=False))
else:
    print("Warning: next_step_correctness columns not found in df")


# next step correctness non optimal 

In [ ]:
# Complete matches vs next_step_correctness in df_non_optimal
# Extended table showing count_sub (0,1,2,3) vs next_step_correctness for each agent
print("=" * 80)
print("df_non_optimal: count_sub vs next_step_correctness (All Agents)")
print("=" * 80)

if 'next_step_correctness_x' in df_non_optimal.columns and 'next_step_correctness_y' in df_non_optimal.columns and 'next_step_correctness_ours' in df_non_optimal.columns:
    
    # Create comprehensive table for all count_sub values
    summary_data = []
    
    for count_sub_val in [0, 1, 2, 3]:
        df_subset = df_non_optimal[df_non_optimal['count_sub'] == count_sub_val].copy()
        
        if len(df_subset) > 0:
            summary_data.append({
                'count_sub': count_sub_val,
                'Total': len(df_subset),
                'Teacher_Correct': (df_subset['next_step_correctness_x'] == 'Correct').sum(),
                'Teacher_Incorrect': (df_subset['next_step_correctness_x'] == 'Incorrect').sum(),
                'Teacher_Suboptimal': (df_subset['next_step_correctness_x'] == 'Suboptimal').sum(),
                'Judge_Correct': (df_subset['next_step_correctness_y'] == 'Correct').sum(),
                'Judge_Incorrect': (df_subset['next_step_correctness_y'] == 'Incorrect').sum(),
                'Judge_Suboptimal': (df_subset['next_step_correctness_y'] == 'Suboptimal').sum(),
                'Ours_Correct': (df_subset['next_step_correctness_ours'] == 'Correct').sum(),
                'Ours_Incorrect': (df_subset['next_step_correctness_ours'] == 'Incorrect').sum(),
                'Ours_Suboptimal': (df_subset['next_step_correctness_ours'] == 'Suboptimal').sum()
            })
        else:
            summary_data.append({
                'count_sub': count_sub_val,
                'Total': 0,
                'Teacher_Correct': 0, 'Teacher_Incorrect': 0, 'Teacher_Suboptimal': 0,
                'Judge_Correct': 0, 'Judge_Incorrect': 0, 'Judge_Suboptimal': 0,
                'Ours_Correct': 0, 'Ours_Incorrect': 0, 'Ours_Suboptimal': 0
            })
    
    summary_df = pd.DataFrame(summary_data)
    
    # Display the table
    print("\n" + "=" * 80)
    print("Summary Table: count_sub vs next_step_correctness")
    print("=" * 80)
    print(summary_df.to_string(index=False))
    
    # Also show breakdown by count_sub description
    print("\n" + "=" * 80)
    print("count_sub Description:")
    print("  0 = no match")
    print("  1 = step match only")
    print("  2 = step + rule match")
    print("  3 = complete match (step + rule + parents)")
    print("=" * 80)
else:
    print("Warning: next_step_correctness columns not found in df_non_optimal")


# F1 Optimal/non optimal predictions

In [ ]:
print("optimal steps: ", df[df['student_count'] == 1].shape[0])
print("non optimal steps: ", df_non_optimal[(df_non_optimal['count_sub'] == 3)].shape[0])

print(df_non_optimal[df_non_optimal['count_sub'].isin([0, 1, 2])].shape[0])



In [ ]:

from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

# Filter to only optimal steps
df_optimal = df[df['student_count'] == 1].copy()

# Ground truth: all are optimal (1)
y_true = np.array([1] * len(df_optimal))

# Predictions: 'Correct' = 1, others = 0
agents = {
    'Teacher': 'next_step_correctness_x',
    'Judge': 'next_step_correctness_y',
    'Ours': 'next_step_correctness_ours'
}

# Bootstrap parameters
n_bootstrap = 1000
confidence_level = 0.95
alpha = 1 - confidence_level

results = []
all_f1_scores = []  # Collect all F1 scores for averaging

for agent_name, col in agents.items():
    y_pred = (df_optimal[col] == 'Correct').astype(int).values
    
    # Original metrics
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    all_f1_scores.append(f1)  # Collect F1 for averaging
    
    # Bootstrap for CI
    n = len(y_true)
    precision_boot = []
    recall_boot = []
    f1_boot = []
    
    np.random.seed(42)  # For reproducibility
    for _ in range(n_bootstrap):
        indices = np.random.choice(n, size=n, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        
        precision_boot.append(precision_score(y_true_boot, y_pred_boot, zero_division=0))
        recall_boot.append(recall_score(y_true_boot, y_pred_boot, zero_division=0))
        f1_boot.append(f1_score(y_true_boot, y_pred_boot, zero_division=0))
    
    # Compute CI (percentile method)
    precision_ci_lower = np.percentile(precision_boot, (alpha/2) * 100)
    precision_ci_upper = np.percentile(precision_boot, (1 - alpha/2) * 100)
    
    recall_ci_lower = np.percentile(recall_boot, (alpha/2) * 100)
    recall_ci_upper = np.percentile(recall_boot, (1 - alpha/2) * 100)
    
    f1_ci_lower = np.percentile(f1_boot, (alpha/2) * 100)
    f1_ci_upper = np.percentile(f1_boot, (1 - alpha/2) * 100)
    
    results.append({
        'Agent': agent_name,
        'Precision': precision,
        'Precision_CI_Lower': precision_ci_lower,
        'Precision_CI_Upper': precision_ci_upper,
        'Recall': recall,
        'Recall_CI_Lower': recall_ci_lower,
        'Recall_CI_Upper': recall_ci_upper,
        'F1_Score': f1,
        'F1_CI_Lower': f1_ci_lower,
        'F1_CI_Upper': f1_ci_upper
    })
    
    print(f"{agent_name}:")
    print(f"  Precision: {precision:.4f} [{precision_ci_lower:.4f}, {precision_ci_upper:.4f}]")
    print(f"  Recall:    {recall:.4f} [{recall_ci_lower:.4f}, {recall_ci_upper:.4f}]")
    print(f"  F1 Score:  {f1:.4f} [{f1_ci_lower:.4f}, {f1_ci_upper:.4f}]")
    print()

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# ============================================================================
# COMPUTE AVERAGE F1 ACROSS ALL AGENTS
# ============================================================================
avg_f1 = np.mean(all_f1_scores)
print("\n" + "="*80)
print(f"Average F1 Score across all agents: {avg_f1:.4f}")
print("="*80)
print(f"Individual F1 scores: {[f'{f:.4f}' for f in all_f1_scores]}")
print(f"Average: {avg_f1:.4f}")

In [37]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

# Compact F1 Score Summary for Teacher, Judge, and Ours
agents = {
    'Teacher': 'next_step_correctness_x',
    'Judge': 'next_step_correctness_y', 
    'Ours': 'next_step_correctness_ours'
}

# Bootstrap parameters
n_bootstrap = 1000
confidence_level = 0.95
alpha = 1 - confidence_level

results = []

for agent_name, pred_col in agents.items():
    # Incorrect: count_sub == 0
    tp_incorrect = ((df_non_optimal[pred_col] == "Incorrect") & (df_non_optimal["count_sub"] == 0)).sum()
    predicted_incorrect = (df_non_optimal[pred_col] == "Incorrect").sum()
    actual_incorrect = (df_non_optimal["count_sub"] == 0).sum()
    
    precision_incorrect = round(tp_incorrect / predicted_incorrect if predicted_incorrect > 0 else 0, 2)
    recall_incorrect = round(tp_incorrect / actual_incorrect if actual_incorrect > 0 else 0, 2)
    f1_incorrect = round(2 * (precision_incorrect * recall_incorrect) / (precision_incorrect + recall_incorrect) if (precision_incorrect + recall_incorrect) > 0 else 0, 2)
    
    # Bootstrap CI for Incorrect
    y_true_incorrect = (df_non_optimal["count_sub"] == 0).astype(int).values
    y_pred_incorrect = (df_non_optimal[pred_col] == "Incorrect").astype(int).values
    
    precision_incorrect_boot = []
    recall_incorrect_boot = []
    f1_incorrect_boot = []
    
    np.random.seed(42)
    n_incorrect = len(y_true_incorrect)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_incorrect, size=n_incorrect, replace=True)
        y_true_boot = y_true_incorrect[indices]
        y_pred_boot = y_pred_incorrect[indices]
        
        # Precision: TP / (TP + FP)
        tp = ((y_true_boot == 1) & (y_pred_boot == 1)).sum()
        fp = ((y_true_boot == 0) & (y_pred_boot == 1)).sum()
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Recall: TP / (TP + FN)
        fn = ((y_true_boot == 1) & (y_pred_boot == 0)).sum()
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # F1
        f = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        
        precision_incorrect_boot.append(p)
        recall_incorrect_boot.append(r)
        f1_incorrect_boot.append(f)
    
    precision_incorrect_ci_lower = round(np.percentile(precision_incorrect_boot, (alpha/2) * 100), 2)
    precision_incorrect_ci_upper = round(np.percentile(precision_incorrect_boot, (1 - alpha/2) * 100), 2)
    recall_incorrect_ci_lower = round(np.percentile(recall_incorrect_boot, (alpha/2) * 100), 2)
    recall_incorrect_ci_upper = round(np.percentile(recall_incorrect_boot, (1 - alpha/2) * 100), 2)
    f1_incorrect_ci_lower = round(np.percentile(f1_incorrect_boot, (alpha/2) * 100), 2)
    f1_incorrect_ci_upper = round(np.percentile(f1_incorrect_boot, (1 - alpha/2) * 100), 2)
    
    # Suboptimal: count_sub == 3
    tp_suboptimal = ((df_non_optimal[pred_col] == "Suboptimal") & (df_non_optimal["count_sub"] == 3)).sum()
    predicted_suboptimal = (df_non_optimal[pred_col] == "Suboptimal").sum()
    actual_suboptimal = (df_non_optimal["count_sub"] == 3).sum()
    
    precision_suboptimal = round(tp_suboptimal / predicted_suboptimal if predicted_suboptimal > 0 else 0, 2)
    recall_suboptimal = round(tp_suboptimal / actual_suboptimal if actual_suboptimal > 0 else 0, 2)
    f1_suboptimal = round(2 * (precision_suboptimal * recall_suboptimal) / (precision_suboptimal + recall_suboptimal) if (precision_suboptimal + recall_suboptimal) > 0 else 0, 2)
    
    # Bootstrap CI for Suboptimal
    y_true_suboptimal = (df_non_optimal["count_sub"] == 3).astype(int).values
    y_pred_suboptimal = (df_non_optimal[pred_col] == "Suboptimal").astype(int).values
    
    precision_suboptimal_boot = []
    recall_suboptimal_boot = []
    f1_suboptimal_boot = []
    
    np.random.seed(42)
    n_suboptimal = len(y_true_suboptimal)
    for _ in range(n_bootstrap):
        indices = np.random.choice(n_suboptimal, size=n_suboptimal, replace=True)
        y_true_boot = y_true_suboptimal[indices]
        y_pred_boot = y_pred_suboptimal[indices]
        
        # Precision: TP / (TP + FP)
        tp = ((y_true_boot == 1) & (y_pred_boot == 1)).sum()
        fp = ((y_true_boot == 0) & (y_pred_boot == 1)).sum()
        p = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Recall: TP / (TP + FN)
        fn = ((y_true_boot == 1) & (y_pred_boot == 0)).sum()
        r = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # F1
        f = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
        
        precision_suboptimal_boot.append(p)
        recall_suboptimal_boot.append(r)
        f1_suboptimal_boot.append(f)
    
    precision_suboptimal_ci_lower = round(np.percentile(precision_suboptimal_boot, (alpha/2) * 100), 2)
    precision_suboptimal_ci_upper = round(np.percentile(precision_suboptimal_boot, (1 - alpha/2) * 100), 2)
    recall_suboptimal_ci_lower = round(np.percentile(recall_suboptimal_boot, (alpha/2) * 100), 2)
    recall_suboptimal_ci_upper = round(np.percentile(recall_suboptimal_boot, (1 - alpha/2) * 100), 2)
    f1_suboptimal_ci_lower = round(np.percentile(f1_suboptimal_boot, (alpha/2) * 100), 2)
    f1_suboptimal_ci_upper = round(np.percentile(f1_suboptimal_boot, (1 - alpha/2) * 100), 2)
    
    results.append({
        'Agent': agent_name,
        'Class': 'Incorrect',
        'Precision': precision_incorrect,
        'Precision_CI': f"[{precision_incorrect_ci_lower}, {precision_incorrect_ci_upper}]",
        'Recall': recall_incorrect,
        'Recall_CI': f"[{recall_incorrect_ci_lower}, {recall_incorrect_ci_upper}]",
        'F1': f1_incorrect,
        'F1_CI': f"[{f1_incorrect_ci_lower}, {f1_incorrect_ci_upper}]"
    })
    results.append({
        'Agent': agent_name,
        'Class': 'Suboptimal',
        'Precision': precision_suboptimal,
        'Precision_CI': f"[{precision_suboptimal_ci_lower}, {precision_suboptimal_ci_upper}]",
        'Recall': recall_suboptimal,
        'Recall_CI': f"[{recall_suboptimal_ci_lower}, {recall_suboptimal_ci_upper}]",
        'F1': f1_suboptimal,
        'F1_CI': f"[{f1_suboptimal_ci_lower}, {f1_suboptimal_ci_upper}]"
    })

# Print summary
summary_df = pd.DataFrame(results)
print("\n" + "="*80)
print("F1 Score Summary with 95% Confidence Intervals")
print("="*80)
print(summary_df.to_string(index=False))
print("\nF1 Scores by Agent and Class:")
print(summary_df.pivot(index='Agent', columns='Class', values='F1').round(3))


F1 Score Summary with 95% Confidence Intervals
  Agent      Class  Precision Precision_CI  Recall    Recall_CI   F1        F1_CI
Teacher  Incorrect       0.35   [0.3, 0.4]    0.78 [0.72, 0.84] 0.48 [0.43, 0.54]
Teacher Suboptimal       0.11  [0.0, 0.38]    0.02  [0.0, 0.08] 0.03  [0.0, 0.13]
  Judge  Incorrect       0.38 [0.33, 0.42]    0.97 [0.94, 0.99] 0.55 [0.49, 0.59]
  Judge Suboptimal       0.00   [0.0, 0.0]    0.00   [0.0, 0.0] 0.00   [0.0, 0.0]
   Ours  Incorrect       0.36  [0.3, 0.42]    0.59  [0.5, 0.66] 0.45 [0.38, 0.51]
   Ours Suboptimal       0.13 [0.06, 0.21]    0.24 [0.12, 0.38] 0.17 [0.08, 0.25]

F1 Scores by Agent and Class:
Class    Incorrect  Suboptimal
Agent                         
Judge         0.55        0.00
Ours          0.45        0.17
Teacher       0.48        0.03


# complaxity and other features for next step assessment 

In [38]:
df["known_expressions_length"] = df["known_expressions"].apply(
    lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0
)
print("Tutor complexity: ",df[
    (df["next_step_correctness_x"] == "Correct") & 
    (df["student_count"] == 1)
]["KG_complexity"].mean())
# print("Tutor complexity: ",df[
#     (df["next_step_correctness_x"] == "Correct") & 
#     (df["student_count"] == 1)
# ]["KG_distance"].mean())
# print("Tutor complexity: ",df[
#     (df["next_step_correctness_x"] == "Correct") & 
#     (df["student_count"] == 1)
# ]["known_expressions_length"].mean())

print("Tutor complexity: ",df[
    (df["next_step_correctness_y"] == "Correct") & 
    (df["student_count"] == 1)
]["KG_complexity"].mean())
# print("Tutor complexity: ",df[
#     (df["next_step_correctness_y"] == "Correct") & 
#     (df["student_count"] == 1)
# ]["KG_distance"].mean())
# print("Tutor complexity: ",df[
#     (df["next_step_correctness_y"] == "Correct") & 
#     (df["student_count"] == 1)
# ]["known_expressions_length"].mean())


print("Tutor complexity: ",df[
    (df["next_step_correctness_ours"] == "Correct") & 
    (df["student_count"] == 1)
]["KG_complexity"].mean())
# print("Tutor complexity: ",df[
#     (df["next_step_correctness_ours"] == "Correct") & 
#     (df["student_count"] == 1)
# ]["KG_distance"].mean())
# print("Tutor complexity: ",df[
#     (df["next_step_correctness_ours"] == "Correct") & 
#     (df["student_count"] == 1)
# ]["known_expressions_length"].mean())

Tutor complexity:  1.896551724137931
Tutor complexity:  2.0697674418604652
Tutor complexity:  1.8644067796610169


In [39]:
df_non_optimal["known_expressions_length"] = df_non_optimal["known_expressions"].apply(
    lambda x: len(ast.literal_eval(x)) if isinstance(x, str) else 0
)

print("Tutor complexity suboptimal: ",df_non_optimal[
    (df_non_optimal["next_step_correctness_x"] == "Suboptimal") & 
    (df_non_optimal["count_sub"] == 3)
]["KG_complexity"].mean())
# print("known_expressions_length: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_x"] == "Suboptimal") & 
#     (df_non_optimal["count_sub"] == 3)
# ]["known_expressions_length"].mean())
# print("distance: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_x"] == "Suboptimal") & 
#     (df_non_optimal["count_sub"] == 3)
# ]["KG_distance"].mean())
print("Tutor complexity incorrect: ",df_non_optimal[
    (df_non_optimal["next_step_correctness_x"] == "Incorrect") & 
    (df_non_optimal["count_sub"] == 0)
]["KG_complexity"].mean())
# print("known_expressions_length: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_x"] == "Incorrect") & 
#     (df_non_optimal["count_sub"] == 0)
# ]["known_expressions_length"].mean())
# print("distance: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_x"] == "Incorrect") & 
#     (df_non_optimal["count_sub"] == 0)
# ]["KG_distance"].mean())


print("Judge complexity suboptimal: ",df_non_optimal[
    (df_non_optimal["next_step_correctness_y"] == "Suboptimal") & 
    (df_non_optimal["count_sub"] == 3)
]["KG_complexity"].mean())
# print("known_expressions_length: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_y"] == "Suboptimal") & 
#     (df_non_optimal["count_sub"] == 3)
# ]["known_expressions_length"].mean())
# print("distance: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_x"] == "Suboptimal") & 
#     (df_non_optimal["count_sub"] == 3)
# ]["KG_distance"].mean())
print("Judge complexity incorrect: ",df_non_optimal[
    (df_non_optimal["next_step_correctness_y"] == "Incorrect") & 
    (df_non_optimal["count_sub"] == 0)
]["KG_complexity"].mean())
# print("known_expressions_length: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_y"] == "Incorrect") & 
#     (df_non_optimal["count_sub"] == 0)
# ]["known_expressions_length"].mean())
# print("distance: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_y"] == "Incorrect") & 
#     (df_non_optimal["count_sub"] == 0)
# ]["KG_distance"].mean())

print("Ours complexity suboptimal: ",df_non_optimal[
    (df_non_optimal["next_step_correctness_ours"] == "Suboptimal") & 
    (df_non_optimal["count_sub"] == 3)
]["KG_complexity"].mean())
# print("known_expressions_length: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_ours"] == "Suboptimal") & 
#     (df_non_optimal["count_sub"] == 3)
# ]["known_expressions_length"].mean())
# print("distance: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_x"] == "Suboptimal") & 
#     (df_non_optimal["count_sub"] == 3)
# ]["KG_distance"].mean())
print("Ours complexity incorrect: ",df_non_optimal[
    (df_non_optimal["next_step_correctness_ours"] == "Incorrect") & 
    (df_non_optimal["count_sub"] == 0)
]["KG_complexity"].mean())
# print("known_expressions_length: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_ours"] == "Incorrect") & 
#     (df_non_optimal["count_sub"] == 0)
# ]["known_expressions_length"].mean())
# print("distance: ",df_non_optimal[
#     (df_non_optimal["next_step_correctness_ours"] == "Incorrect") & 
#     (df_non_optimal["count_sub"] == 0)
# ]["KG_distance"].mean())

Tutor complexity suboptimal:  2.0
Tutor complexity incorrect:  3.3828125
Judge complexity suboptimal:  nan
Judge complexity incorrect:  3.138364779874214
Ours complexity suboptimal:  3.1
Ours complexity incorrect:  3.3541666666666665


In [41]:
import pandas as pd
import numpy as np
from collections import Counter

# Define agents
agents = {
    'Tutor': 'tutor_prediction_column',  # Replace with actual
    'Teacher': 'next_step_correctness_x',
    'Judge': 'next_step_correctness_y'
}

print("="*80)
print("KG_COMPLEXITY ANALYSIS (POOLED ACROSS Tu/Te/J)")
print("="*80)

results = []

# ============================================================================
# 1. OPTIMAL (student_count == 1)
# ============================================================================
print("\nOptimal (student_count == 1):")
print("-" * 80)

df_optimal = df[df['student_count'] == 1].copy()

# For each instance, collect all agent predictions
instance_classifications = {}

for idx in df_optimal.index:
    predictions = []
    for agent_name, pred_col in agents.items():
        if pred_col in df_optimal.columns:
            pred = df_optimal.loc[idx, pred_col]
            if pd.notna(pred):
                # Correct if predicts "Correct", incorrect otherwise
                is_correct = (pred == 'Correct')
                predictions.append(is_correct)
    
    if len(predictions) > 0:
        # Use majority vote (or you can use: all must agree, or any must agree)
        # Majority vote:
        correct_votes = sum(predictions)
        instance_classifications[idx] = correct_votes > len(predictions) / 2

# Separate into correct and incorrect
correct_indices = [idx for idx, is_correct in instance_classifications.items() if is_correct]
incorrect_indices = [idx for idx, is_correct in instance_classifications.items() if not is_correct]

correct_count = len(correct_indices)
incorrect_count = len(incorrect_indices)

print(f"Total optimal instances: {len(df_optimal)}")
print(f"Correctly classified (majority vote): {correct_count}")
print(f"Incorrectly classified (majority vote): {incorrect_count}")
print(f"Total: {correct_count + incorrect_count} (should equal {len(df_optimal)})")

if 'KG_complexity' in df_optimal.columns:
    correct_complexity_mean = df_optimal.loc[correct_indices, 'KG_complexity'].mean() if correct_count > 0 else np.nan
    incorrect_complexity_mean = df_optimal.loc[incorrect_indices, 'KG_complexity'].mean() if incorrect_count > 0 else np.nan
else:
    correct_complexity_mean = np.nan
    incorrect_complexity_mean = np.nan

print(f"  - Correctly classified: mean = {correct_complexity_mean:.4f}, n = {correct_count}")
print(f"  - Incorrectly classified: mean = {incorrect_complexity_mean:.4f}, n = {incorrect_count}")

results.append({
    'Category': 'Optimal',
    'Correctly_Classified_Mean_KG_Complexity': correct_complexity_mean,
    'Correctly_Classified_n': correct_count,
    'Incorrectly_Classified_Mean_KG_Complexity': incorrect_complexity_mean,
    'Incorrectly_Classified_n': incorrect_count
})

# ============================================================================
# 2. VALID ALT (count_sub == 3)
# ============================================================================
print("\nValid Alt (count_sub == 3):")
print("-" * 80)

df_valid_alt = df_non_optimal[df_non_optimal['count_sub'] == 3].copy()

instance_classifications = {}

for idx in df_valid_alt.index:
    predictions = []
    for agent_name, pred_col in agents.items():
        if pred_col in df_valid_alt.columns:
            pred = df_valid_alt.loc[idx, pred_col]
            if pd.notna(pred):
                # Correct if predicts "Suboptimal"
                is_correct = (pred == 'Suboptimal')
                predictions.append(is_correct)
    
    if len(predictions) > 0:
        correct_votes = sum(predictions)
        instance_classifications[idx] = correct_votes > len(predictions) / 2

correct_indices = [idx for idx, is_correct in instance_classifications.items() if is_correct]
incorrect_indices = [idx for idx, is_correct in instance_classifications.items() if not is_correct]

correct_count = len(correct_indices)
incorrect_count = len(incorrect_indices)

print(f"Total valid alt instances: {len(df_valid_alt)}")
print(f"Correctly classified (majority vote): {correct_count}")
print(f"Incorrectly classified (majority vote): {incorrect_count}")
print(f"Total: {correct_count + incorrect_count} (should equal {len(df_valid_alt)})")

if 'KG_complexity' in df_valid_alt.columns:
    correct_complexity_mean = df_valid_alt.loc[correct_indices, 'KG_complexity'].mean() if correct_count > 0 else np.nan
    incorrect_complexity_mean = df_valid_alt.loc[incorrect_indices, 'KG_complexity'].mean() if incorrect_count > 0 else np.nan
else:
    correct_complexity_mean = np.nan
    incorrect_complexity_mean = np.nan

print(f"  - Correctly classified: mean = {correct_complexity_mean:.4f}, n = {correct_count}")
print(f"  - Incorrectly classified: mean = {incorrect_complexity_mean:.4f}, n = {incorrect_count}")

results.append({
    'Category': 'Valid Alt',
    'Correctly_Classified_Mean_KG_Complexity': correct_complexity_mean,
    'Correctly_Classified_n': correct_count,
    'Incorrectly_Classified_Mean_KG_Complexity': incorrect_complexity_mean,
    'Incorrectly_Classified_n': incorrect_count
})

# ============================================================================
# 3. INCORRECT (count_sub in [0, 1, 2])
# ============================================================================
print("\nIncorrect (count_sub in [0, 1, 2]):")
print("-" * 80)

df_incorrect = df_non_optimal[df_non_optimal['count_sub'].isin([0, 1, 2])].copy()

instance_classifications = {}

for idx in df_incorrect.index:
    predictions = []
    for agent_name, pred_col in agents.items():
        if pred_col in df_incorrect.columns:
            pred = df_incorrect.loc[idx, pred_col]
            if pd.notna(pred):
                # Correct if predicts "Incorrect"
                is_correct = (pred == 'Incorrect')
                predictions.append(is_correct)
    
    if len(predictions) > 0:
        correct_votes = sum(predictions)
        instance_classifications[idx] = correct_votes > len(predictions) / 2

correct_indices = [idx for idx, is_correct in instance_classifications.items() if is_correct]
incorrect_indices = [idx for idx, is_correct in instance_classifications.items() if not is_correct]

correct_count = len(correct_indices)
incorrect_count = len(incorrect_indices)

print(f"Total incorrect instances: {len(df_incorrect)}")
print(f"Correctly classified (majority vote): {correct_count}")
print(f"Incorrectly classified (majority vote): {incorrect_count}")
print(f"Total: {correct_count + incorrect_count} (should equal {len(df_incorrect)})")

if 'KG_complexity' in df_incorrect.columns:
    correct_complexity_mean = df_incorrect.loc[correct_indices, 'KG_complexity'].mean() if correct_count > 0 else np.nan
    incorrect_complexity_mean = df_incorrect.loc[incorrect_indices, 'KG_complexity'].mean() if incorrect_count > 0 else np.nan
else:
    correct_complexity_mean = np.nan
    incorrect_complexity_mean = np.nan

print(f"  - Correctly classified: mean = {correct_complexity_mean:.4f}, n = {correct_count}")
print(f"  - Incorrectly classified: mean = {incorrect_complexity_mean:.4f}, n = {incorrect_count}")

results.append({
    'Category': 'Incorrect',
    'Correctly_Classified_Mean_KG_Complexity': correct_complexity_mean,
    'Correctly_Classified_n': correct_count,
    'Incorrectly_Classified_Mean_KG_Complexity': incorrect_complexity_mean,
    'Incorrectly_Classified_n': incorrect_count
})

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

total_n = results_df['Correctly_Classified_n'].sum() + results_df['Incorrectly_Classified_n'].sum()
print(f"\nTotal instances: {total_n} (should equal 516)")

KG_COMPLEXITY ANALYSIS (POOLED ACROSS Tu/Te/J)

Optimal (student_count == 1):
--------------------------------------------------------------------------------
Total optimal instances: 59
Correctly classified (majority vote): 43
Incorrectly classified (majority vote): 16
Total: 59 (should equal 59)
  - Correctly classified: mean = 2.0698, n = 43
  - Incorrectly classified: mean = 1.3125, n = 16

Valid Alt (count_sub == 3):
--------------------------------------------------------------------------------
Total valid alt instances: 41
Correctly classified (majority vote): 0
Incorrectly classified (majority vote): 41
Total: 41 (should equal 41)
  - Correctly classified: mean = nan, n = 0
  - Incorrectly classified: mean = 2.8537, n = 41

Incorrect (count_sub in [0, 1, 2]):
--------------------------------------------------------------------------------
Total incorrect instances: 407
Correctly classified (majority vote): 320
Incorrectly classified (majority vote): 87
Total: 407 (should equal

In [42]:
import pandas as pd

print("="*80)
print("CONDITIONAL PROBABILITIES: MISLABELING")
print("="*80)
print("Case 1: P(labeled as 'Suboptimal' | count_sub in [0,1,2])")
print("Case 2: P(labeled as 'Incorrect' | count_sub == 3)")
print("Treating 'Correct' as 'Incorrect'")
print("="*80)

agents = {
    'Teacher': 'next_step_correctness_x',
    'Judge': 'next_step_correctness_y',
    'Ours': 'next_step_correctness_ours'
}

# ============================================================================
# SUMMARY TABLE FOR EACH AGENT
# ============================================================================
print("\n" + "="*80)
print("SUMMARY TABLE BY AGENT")
print("="*80)

agent_summaries = []

for agent_name, pred_col in agents.items():
    if pred_col not in df_non_optimal.columns:
        continue
    
    # Create binary: Suboptimal vs Incorrect (treating Correct as Incorrect)
    df_agent = df_non_optimal.copy()
    df_agent['label_binary'] = df_agent[pred_col].apply(
        lambda x: 'Suboptimal' if x == 'Suboptimal' else 'Incorrect'
    )
    
    # Case 1: count_sub in [0,1,2]
    case1 = df_agent[df_agent['count_sub'].isin([0, 1, 2])]
    if len(case1) > 0:
        p_sub_case1 = (case1['label_binary'] == 'Suboptimal').sum() / len(case1) * 100
        p_inc_case1 = (case1['label_binary'] == 'Incorrect').sum() / len(case1) * 100
    else:
        p_sub_case1 = 0
        p_inc_case1 = 0
    
    # Case 2: count_sub == 3
    case2 = df_agent[df_agent['count_sub'] == 3]
    if len(case2) > 0:
        p_sub_case2 = (case2['label_binary'] == 'Suboptimal').sum() / len(case2) * 100
        p_inc_case2 = (case2['label_binary'] == 'Incorrect').sum() / len(case2) * 100
    else:
        p_sub_case2 = 0
        p_inc_case2 = 0
    
    agent_summaries.append({
        'Agent': agent_name,
        'count_sub': '0,1,2',
        'P(Suboptimal)': round(p_sub_case1, 2),
        'P(Incorrect)': round(p_inc_case1, 2),
        'n': len(case1)
    })
    
    agent_summaries.append({
        'Agent': agent_name,
        'count_sub': '3',
        'P(Suboptimal)': round(p_sub_case2, 2),
        'P(Incorrect)': round(p_inc_case2, 2),
        'n': len(case2)
    })

agent_summary_df = pd.DataFrame(agent_summaries)
print(agent_summary_df.to_string(index=False))

# ============================================================================
# POOLED SUMMARY ACROSS ALL AGENTS
# ============================================================================
print("\n" + "="*80)
print("POOLED SUMMARY ACROSS ALL AGENTS")
print("="*80)

# Collect all predictions across all agents
all_predictions = []

for agent_name, pred_col in agents.items():
    if pred_col not in df_non_optimal.columns:
        continue
    
    # Create binary: Suboptimal vs Incorrect (treating Correct as Incorrect)
    df_agent = df_non_optimal.copy()
    df_agent['label_binary'] = df_agent[pred_col].apply(
        lambda x: 'Suboptimal' if x == 'Suboptimal' else 'Incorrect'
    )
    df_agent['agent'] = agent_name
    
    all_predictions.append(df_agent[['count_sub', 'label_binary', 'agent']])

# Combine all agents
combined = pd.concat(all_predictions, ignore_index=True)

# Case 1: count_sub in [0,1,2]
case1_data = combined[combined['count_sub'].isin([0, 1, 2])]
if len(case1_data) > 0:
    prob_suboptimal_case1 = (case1_data['label_binary'] == 'Suboptimal').sum() / len(case1_data) * 100
    prob_incorrect_case1 = (case1_data['label_binary'] == 'Incorrect').sum() / len(case1_data) * 100
else:
    prob_suboptimal_case1 = 0
    prob_incorrect_case1 = 0

# Case 2: count_sub == 3
case2_data = combined[combined['count_sub'] == 3]
if len(case2_data) > 0:
    prob_suboptimal_case2 = (case2_data['label_binary'] == 'Suboptimal').sum() / len(case2_data) * 100
    prob_incorrect_case2 = (case2_data['label_binary'] == 'Incorrect').sum() / len(case2_data) * 100
else:
    prob_suboptimal_case2 = 0
    prob_incorrect_case2 = 0

pooled_summary = pd.DataFrame({
    'count_sub': ['0,1,2 (collapsed)', '3'],
    'P(Suboptimal)': [round(prob_suboptimal_case1, 2), round(prob_suboptimal_case2, 2)],
    'P(Incorrect)': [round(prob_incorrect_case1, 2), round(prob_incorrect_case2, 2)],
    'n_labels': [len(case1_data), len(case2_data)]
})

print(pooled_summary.to_string(index=False))

# ============================================================================
# DETAILED BREAKDOWN
# ============================================================================
print("\n" + "="*80)
print("DETAILED BREAKDOWN")
print("="*80)

print("\nCASE 1: count_sub in [0,1,2] labeled as 'Suboptimal'")
print("-" * 80)
print(f"Total labels (pooled): {len(case1_data)}")
print(f"P(labeled as 'Suboptimal' | count_sub in [0,1,2]) = {prob_suboptimal_case1:.2f}%")
print(f"P(labeled as 'Incorrect' | count_sub in [0,1,2]) = {prob_incorrect_case1:.2f}%")

print("\nBreakdown by count_sub (pooled):")
for count_val in [0, 1, 2]:
    subset = case1_data[case1_data['count_sub'] == count_val]
    if len(subset) > 0:
        p_sub = (subset['label_binary'] == 'Suboptimal').sum() / len(subset) * 100
        p_inc = (subset['label_binary'] == 'Incorrect').sum() / len(subset) * 100
        print(f"  count_sub={count_val}: Suboptimal={p_sub:.2f}%, Incorrect={p_inc:.2f}% (n={len(subset)} labels)")

print("\nCASE 2: count_sub == 3 labeled as 'Incorrect'")
print("-" * 80)
print(f"Total labels (pooled): {len(case2_data)}")
print(f"P(labeled as 'Suboptimal' | count_sub == 3) = {prob_suboptimal_case2:.2f}%")
print(f"P(labeled as 'Incorrect' | count_sub == 3) = {prob_incorrect_case2:.2f}%")

CONDITIONAL PROBABILITIES: MISLABELING
Case 1: P(labeled as 'Suboptimal' | count_sub in [0,1,2])
Case 2: P(labeled as 'Incorrect' | count_sub == 3)
Treating 'Correct' as 'Incorrect'

SUMMARY TABLE BY AGENT
  Agent count_sub  P(Suboptimal)  P(Incorrect)   n
Teacher     0,1,2           1.97         98.03 407
Teacher         3           2.44         97.56  41
  Judge     0,1,2           0.74         99.26 407
  Judge         3           0.00        100.00  41
   Ours     0,1,2          16.71         83.29 407
   Ours         3          24.39         75.61  41

POOLED SUMMARY ACROSS ALL AGENTS
        count_sub  P(Suboptimal)  P(Incorrect)  n_labels
0,1,2 (collapsed)           6.47         93.53      1221
                3           8.94         91.06       123

DETAILED BREAKDOWN

CASE 1: count_sub in [0,1,2] labeled as 'Suboptimal'
--------------------------------------------------------------------------------
Total labels (pooled): 1221
P(labeled as 'Suboptimal' | count_sub in [0,1,2])

In [43]:
import pandas as pd
import numpy as np

print("="*80)
print("CONDITIONAL PROBABILITIES: MISLABELING")
print("="*80)
print("Case 1: P(labeled as 'Suboptimal' | count_sub in [0,1,2])")
print("Case 2: P(labeled as 'Incorrect' | count_sub == 3)")
print("Treating 'Correct' as 'Incorrect'")
print("="*80)

agents = {
    'Teacher': 'next_step_correctness_x',
    'Judge': 'next_step_correctness_y',
    'Ours': 'next_step_correctness_ours'
}

# ============================================================================
# SUMMARY TABLE FOR EACH AGENT
# ============================================================================
print("\n" + "="*80)
print("SUMMARY TABLE BY AGENT")
print("="*80)

agent_summaries = []

for agent_name, pred_col in agents.items():
    if pred_col not in df_non_optimal.columns:
        continue
    
    # Create binary: Suboptimal vs Incorrect (treating Correct as Incorrect)
    df_agent = df_non_optimal.copy()
    df_agent['label_binary'] = df_agent[pred_col].apply(
        lambda x: 'Suboptimal' if x == 'Suboptimal' else 'Incorrect'
    )
    
    # Case 1: count_sub in [0,1,2]
    case1 = df_agent[df_agent['count_sub'].isin([0, 1, 2])]
    if len(case1) > 0:
        p_sub_case1 = (case1['label_binary'] == 'Suboptimal').sum() / len(case1) * 100
        p_inc_case1 = (case1['label_binary'] == 'Incorrect').sum() / len(case1) * 100
    else:
        p_sub_case1 = 0
        p_inc_case1 = 0
    
    # Case 2: count_sub == 3
    case2 = df_agent[df_agent['count_sub'] == 3]
    if len(case2) > 0:
        p_sub_case2 = (case2['label_binary'] == 'Suboptimal').sum() / len(case2) * 100
        p_inc_case2 = (case2['label_binary'] == 'Incorrect').sum() / len(case2) * 100
    else:
        p_sub_case2 = 0
        p_inc_case2 = 0
    
    agent_summaries.append({
        'Agent': agent_name,
        'count_sub': '0,1,2',
        'P(Suboptimal)': round(p_sub_case1, 2),
        'P(Incorrect)': round(p_inc_case1, 2),
        'n': len(case1)
    })
    
    agent_summaries.append({
        'Agent': agent_name,
        'count_sub': '3',
        'P(Suboptimal)': round(p_sub_case2, 2),
        'P(Incorrect)': round(p_inc_case2, 2),
        'n': len(case2)
    })

agent_summary_df = pd.DataFrame(agent_summaries)
print(agent_summary_df.to_string(index=False))

# ============================================================================
# POOLED SUMMARY ACROSS ALL AGENTS
# ============================================================================
print("\n" + "="*80)
print("POOLED SUMMARY ACROSS ALL AGENTS")
print("="*80)

# Collect all predictions across all agents (including KG_complexity)
all_predictions = []

for agent_name, pred_col in agents.items():
    if pred_col not in df_non_optimal.columns:
        continue
    
    # Create binary: Suboptimal vs Incorrect (treating Correct as Incorrect)
    df_agent = df_non_optimal.copy()
    df_agent['label_binary'] = df_agent[pred_col].apply(
        lambda x: 'Suboptimal' if x == 'Suboptimal' else 'Incorrect'
    )
    df_agent['agent'] = agent_name
    
    # Include KG_complexity if available
    cols_to_include = ['count_sub', 'label_binary', 'agent']
    if 'KG_complexity' in df_agent.columns:
        cols_to_include.append('KG_complexity')
    
    all_predictions.append(df_agent[cols_to_include])

# Combine all agents
combined = pd.concat(all_predictions, ignore_index=True)

# Normalize complexity values (z-score standardization) for fair comparison across groups
if 'KG_complexity' in combined.columns:
    # Compute overall mean and std for normalization
    overall_mean = combined['KG_complexity'].mean()
    overall_std = combined['KG_complexity'].std()
    
    if overall_std > 0:
        combined['KG_complexity_normalized'] = (combined['KG_complexity'] - overall_mean) / overall_std
        print(f"\nComplexity normalization: mean={overall_mean:.2f}, std={overall_std:.2f}")
    else:
        combined['KG_complexity_normalized'] = 0
        print("\nWarning: Standard deviation is 0, normalization not applied")

# Case 1: count_sub in [0,1,2]
case1_data = combined[combined['count_sub'].isin([0, 1, 2])]
if len(case1_data) > 0:
    prob_suboptimal_case1 = (case1_data['label_binary'] == 'Suboptimal').sum() / len(case1_data) * 100
    prob_incorrect_case1 = (case1_data['label_binary'] == 'Incorrect').sum() / len(case1_data) * 100
    
    # Compute mean complexity for each label in case 1 (both raw and normalized)
    if 'KG_complexity' in case1_data.columns:
        case1_suboptimal = case1_data[case1_data['label_binary'] == 'Suboptimal']
        case1_incorrect = case1_data[case1_data['label_binary'] == 'Incorrect']
        
        # Raw means
        mean_complexity_sub_case1 = case1_suboptimal['KG_complexity'].mean() if len(case1_suboptimal) > 0 else np.nan
        mean_complexity_inc_case1 = case1_incorrect['KG_complexity'].mean() if len(case1_incorrect) > 0 else np.nan
        mean_complexity_all_case1 = case1_data['KG_complexity'].mean()
        
        # Normalized means
        if 'KG_complexity_normalized' in case1_data.columns:
            mean_complexity_norm_sub_case1 = case1_suboptimal['KG_complexity_normalized'].mean() if len(case1_suboptimal) > 0 else np.nan
            mean_complexity_norm_inc_case1 = case1_incorrect['KG_complexity_normalized'].mean() if len(case1_incorrect) > 0 else np.nan
            mean_complexity_norm_all_case1 = case1_data['KG_complexity_normalized'].mean()
        else:
            mean_complexity_norm_sub_case1 = np.nan
            mean_complexity_norm_inc_case1 = np.nan
            mean_complexity_norm_all_case1 = np.nan
    else:
        mean_complexity_sub_case1 = np.nan
        mean_complexity_inc_case1 = np.nan
        mean_complexity_all_case1 = np.nan
        mean_complexity_norm_sub_case1 = np.nan
        mean_complexity_norm_inc_case1 = np.nan
        mean_complexity_norm_all_case1 = np.nan
else:
    prob_suboptimal_case1 = 0
    prob_incorrect_case1 = 0
    mean_complexity_sub_case1 = np.nan
    mean_complexity_inc_case1 = np.nan
    mean_complexity_all_case1 = np.nan
    mean_complexity_norm_sub_case1 = np.nan
    mean_complexity_norm_inc_case1 = np.nan
    mean_complexity_norm_all_case1 = np.nan

# Case 2: count_sub == 3
case2_data = combined[combined['count_sub'] == 3]
if len(case2_data) > 0:
    prob_suboptimal_case2 = (case2_data['label_binary'] == 'Suboptimal').sum() / len(case2_data) * 100
    prob_incorrect_case2 = (case2_data['label_binary'] == 'Incorrect').sum() / len(case2_data) * 100
    
    # Compute mean complexity for each label in case 2 (both raw and normalized)
    if 'KG_complexity' in case2_data.columns:
        case2_suboptimal = case2_data[case2_data['label_binary'] == 'Suboptimal']
        case2_incorrect = case2_data[case2_data['label_binary'] == 'Incorrect']
        
        # Raw means
        mean_complexity_sub_case2 = case2_suboptimal['KG_complexity'].mean() if len(case2_suboptimal) > 0 else np.nan
        mean_complexity_inc_case2 = case2_incorrect['KG_complexity'].mean() if len(case2_incorrect) > 0 else np.nan
        mean_complexity_all_case2 = case2_data['KG_complexity'].mean()
        
        # Normalized means
        if 'KG_complexity_normalized' in case2_data.columns:
            mean_complexity_norm_sub_case2 = case2_suboptimal['KG_complexity_normalized'].mean() if len(case2_suboptimal) > 0 else np.nan
            mean_complexity_norm_inc_case2 = case2_incorrect['KG_complexity_normalized'].mean() if len(case2_incorrect) > 0 else np.nan
            mean_complexity_norm_all_case2 = case2_data['KG_complexity_normalized'].mean()
        else:
            mean_complexity_norm_sub_case2 = np.nan
            mean_complexity_norm_inc_case2 = np.nan
            mean_complexity_norm_all_case2 = np.nan
    else:
        mean_complexity_sub_case2 = np.nan
        mean_complexity_inc_case2 = np.nan
        mean_complexity_all_case2 = np.nan
        mean_complexity_norm_sub_case2 = np.nan
        mean_complexity_norm_inc_case2 = np.nan
        mean_complexity_norm_all_case2 = np.nan
else:
    prob_suboptimal_case2 = 0
    prob_incorrect_case2 = 0
    mean_complexity_sub_case2 = np.nan
    mean_complexity_inc_case2 = np.nan
    mean_complexity_all_case2 = np.nan
    mean_complexity_norm_sub_case2 = np.nan
    mean_complexity_norm_inc_case2 = np.nan
    mean_complexity_norm_all_case2 = np.nan

# Create pooled summary with complexity
pooled_summary_data = {
    'count_sub': ['0,1,2 (collapsed)', '3'],
    'P(Suboptimal)': [round(prob_suboptimal_case1, 2), round(prob_suboptimal_case2, 2)],
    'P(Incorrect)': [round(prob_incorrect_case1, 2), round(prob_incorrect_case2, 2)],
    'n_labels': [len(case1_data), len(case2_data)]
}

# Add complexity columns if available (both raw and normalized)
if 'KG_complexity' in combined.columns:
    # Raw complexity
    pooled_summary_data['Mean_Complexity_All'] = [
        round(mean_complexity_all_case1, 2) if not np.isnan(mean_complexity_all_case1) else 'N/A',
        round(mean_complexity_all_case2, 2) if not np.isnan(mean_complexity_all_case2) else 'N/A'
    ]
    pooled_summary_data['Mean_Complexity_Suboptimal'] = [
        round(mean_complexity_sub_case1, 2) if not np.isnan(mean_complexity_sub_case1) else 'N/A',
        round(mean_complexity_sub_case2, 2) if not np.isnan(mean_complexity_sub_case2) else 'N/A'
    ]
    pooled_summary_data['Mean_Complexity_Incorrect'] = [
        round(mean_complexity_inc_case1, 2) if not np.isnan(mean_complexity_inc_case1) else 'N/A',
        round(mean_complexity_inc_case2, 2) if not np.isnan(mean_complexity_inc_case2) else 'N/A'
    ]
    
    # Normalized complexity (z-scores)
    if 'KG_complexity_normalized' in combined.columns:
        pooled_summary_data['Mean_Complexity_Norm_All'] = [
            round(mean_complexity_norm_all_case1, 3) if not np.isnan(mean_complexity_norm_all_case1) else 'N/A',
            round(mean_complexity_norm_all_case2, 3) if not np.isnan(mean_complexity_norm_all_case2) else 'N/A'
        ]
        pooled_summary_data['Mean_Complexity_Norm_Suboptimal'] = [
            round(mean_complexity_norm_sub_case1, 3) if not np.isnan(mean_complexity_norm_sub_case1) else 'N/A',
            round(mean_complexity_norm_sub_case2, 3) if not np.isnan(mean_complexity_norm_sub_case2) else 'N/A'
        ]
        pooled_summary_data['Mean_Complexity_Norm_Incorrect'] = [
            round(mean_complexity_norm_inc_case1, 3) if not np.isnan(mean_complexity_norm_inc_case1) else 'N/A',
            round(mean_complexity_norm_inc_case2, 3) if not np.isnan(mean_complexity_norm_inc_case2) else 'N/A'
        ]

pooled_summary = pd.DataFrame(pooled_summary_data)
print(pooled_summary.to_string(index=False))

# ============================================================================
# DETAILED BREAKDOWN
# ============================================================================
print("\n" + "="*80)
print("DETAILED BREAKDOWN")
print("="*80)

print("\nCASE 1: count_sub in [0,1,2] labeled as 'Suboptimal'")
print("-" * 80)
print(f"Total labels (pooled): {len(case1_data)}")
print(f"P(labeled as 'Suboptimal' | count_sub in [0,1,2]) = {prob_suboptimal_case1:.2f}%")
print(f"P(labeled as 'Incorrect' | count_sub in [0,1,2]) = {prob_incorrect_case1:.2f}%")

# Add complexity information for case 1 (both raw and normalized)
if 'KG_complexity' in case1_data.columns:
    print(f"\nMean KG_Complexity (all): {mean_complexity_all_case1:.2f}" if not np.isnan(mean_complexity_all_case1) else "\nMean KG_Complexity (all): N/A")
    print(f"Mean KG_Complexity (Suboptimal): {mean_complexity_sub_case1:.2f}" if not np.isnan(mean_complexity_sub_case1) else "Mean KG_Complexity (Suboptimal): N/A")
    print(f"Mean KG_Complexity (Incorrect): {mean_complexity_inc_case1:.2f}" if not np.isnan(mean_complexity_inc_case1) else "Mean KG_Complexity (Incorrect): N/A")
    
    # Normalized values
    if 'KG_complexity_normalized' in case1_data.columns:
        print(f"Mean KG_Complexity_Normalized (all): {mean_complexity_norm_all_case1:.3f}" if not np.isnan(mean_complexity_norm_all_case1) else "Mean KG_Complexity_Normalized (all): N/A")
        print(f"Mean KG_Complexity_Normalized (Suboptimal): {mean_complexity_norm_sub_case1:.3f}" if not np.isnan(mean_complexity_norm_sub_case1) else "Mean KG_Complexity_Normalized (Suboptimal): N/A")
        print(f"Mean KG_Complexity_Normalized (Incorrect): {mean_complexity_norm_inc_case1:.3f}" if not np.isnan(mean_complexity_norm_inc_case1) else "Mean KG_Complexity_Normalized (Incorrect): N/A")

print("\nBreakdown by count_sub (pooled):")
for count_val in [0, 1, 2]:
    subset = case1_data[case1_data['count_sub'] == count_val]
    if len(subset) > 0:
        p_sub = (subset['label_binary'] == 'Suboptimal').sum() / len(subset) * 100
        p_inc = (subset['label_binary'] == 'Incorrect').sum() / len(subset) * 100
        
        # Add complexity for this count_sub value (both raw and normalized)
        complexity_info = ""
        if 'KG_complexity' in subset.columns:
            subset_suboptimal = subset[subset['label_binary'] == 'Suboptimal']
            subset_incorrect = subset[subset['label_binary'] == 'Incorrect']
            
            # Raw complexity
            mean_comp_all = subset['KG_complexity'].mean()
            mean_comp_sub = subset_suboptimal['KG_complexity'].mean() if len(subset_suboptimal) > 0 else np.nan
            mean_comp_inc = subset_incorrect['KG_complexity'].mean() if len(subset_incorrect) > 0 else np.nan
            
            complexity_info = f" | Mean Complexity: All={mean_comp_all:.2f}"
            if not np.isnan(mean_comp_sub):
                complexity_info += f", Suboptimal={mean_comp_sub:.2f}"
            if not np.isnan(mean_comp_inc):
                complexity_info += f", Incorrect={mean_comp_inc:.2f}"
            
            # Normalized complexity
            if 'KG_complexity_normalized' in subset.columns:
                mean_comp_norm_all = subset['KG_complexity_normalized'].mean()
                mean_comp_norm_sub = subset_suboptimal['KG_complexity_normalized'].mean() if len(subset_suboptimal) > 0 else np.nan
                mean_comp_norm_inc = subset_incorrect['KG_complexity_normalized'].mean() if len(subset_incorrect) > 0 else np.nan
                
                complexity_info += f" | Norm: All={mean_comp_norm_all:.3f}"
                if not np.isnan(mean_comp_norm_sub):
                    complexity_info += f", Suboptimal={mean_comp_norm_sub:.3f}"
                if not np.isnan(mean_comp_norm_inc):
                    complexity_info += f", Incorrect={mean_comp_norm_inc:.3f}"
        
        print(f"  count_sub={count_val}: Suboptimal={p_sub:.2f}%, Incorrect={p_inc:.2f}% (n={len(subset)} labels){complexity_info}")

print("\nCASE 2: count_sub == 3 labeled as 'Incorrect'")
print("-" * 80)
print(f"Total labels (pooled): {len(case2_data)}")
print(f"P(labeled as 'Suboptimal' | count_sub == 3) = {prob_suboptimal_case2:.2f}%")
print(f"P(labeled as 'Incorrect' | count_sub == 3) = {prob_incorrect_case2:.2f}%")

# Add complexity information for case 2 (both raw and normalized)
if 'KG_complexity' in case2_data.columns:
    print(f"\nMean KG_Complexity (all): {mean_complexity_all_case2:.2f}" if not np.isnan(mean_complexity_all_case2) else "\nMean KG_Complexity (all): N/A")
    print(f"Mean KG_Complexity (Suboptimal): {mean_complexity_sub_case2:.2f}" if not np.isnan(mean_complexity_sub_case2) else "Mean KG_Complexity (Suboptimal): N/A")
    print(f"Mean KG_Complexity (Incorrect): {mean_complexity_inc_case2:.2f}" if not np.isnan(mean_complexity_inc_case2) else "Mean KG_Complexity (Incorrect): N/A")
    
    # Normalized values
    if 'KG_complexity_normalized' in case2_data.columns:
        print(f"Mean KG_Complexity_Normalized (all): {mean_complexity_norm_all_case2:.3f}" if not np.isnan(mean_complexity_norm_all_case2) else "Mean KG_Complexity_Normalized (all): N/A")
        print(f"Mean KG_Complexity_Normalized (Suboptimal): {mean_complexity_norm_sub_case2:.3f}" if not np.isnan(mean_complexity_norm_sub_case2) else "Mean KG_Complexity_Normalized (Suboptimal): N/A")
        print(f"Mean KG_Complexity_Normalized (Incorrect): {mean_complexity_norm_inc_case2:.3f}" if not np.isnan(mean_complexity_norm_inc_case2) else "Mean KG_Complexity_Normalized (Incorrect): N/A")

CONDITIONAL PROBABILITIES: MISLABELING
Case 1: P(labeled as 'Suboptimal' | count_sub in [0,1,2])
Case 2: P(labeled as 'Incorrect' | count_sub == 3)
Treating 'Correct' as 'Incorrect'

SUMMARY TABLE BY AGENT
  Agent count_sub  P(Suboptimal)  P(Incorrect)   n
Teacher     0,1,2           1.97         98.03 407
Teacher         3           2.44         97.56  41
  Judge     0,1,2           0.74         99.26 407
  Judge         3           0.00        100.00  41
   Ours     0,1,2          16.71         83.29 407
   Ours         3          24.39         75.61  41

POOLED SUMMARY ACROSS ALL AGENTS

Complexity normalization: mean=3.25, std=2.66
        count_sub  P(Suboptimal)  P(Incorrect)  n_labels  Mean_Complexity_All  Mean_Complexity_Suboptimal  Mean_Complexity_Incorrect  Mean_Complexity_Norm_All  Mean_Complexity_Norm_Suboptimal  Mean_Complexity_Norm_Incorrect
0,1,2 (collapsed)           6.47         93.53      1221                 3.29                        2.91                       3.32

In [44]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import confusion_matrix, cohen_kappa_score
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("BINARY LOGISTIC REGRESSIONS")
print("="*80)
print("Note: 'Correct' labels are treated as 'Incorrect'")
print("="*80)

# ============================================================================
# DATA PREPARATION
# ============================================================================
print("\n1. DATA PREPARATION")
print("-" * 80)

# Use df_non_optimal which has count_sub
if 'df_non_optimal' not in globals():
    print("ERROR: df_non_optimal not found. Please ensure it's defined.")
else:
    df_work = df_non_optimal.copy()
    
    # Check for count_sub
    if 'count_sub' not in df_work.columns:
        print("ERROR: 'count_sub' not found in df_non_optimal")
    else:
        print("Using df_non_optimal with count_sub column")
        
        # Extract problem_level from problem_number
        if 'problem_number' in df_work.columns:
            df_work['problem_level'] = pd.to_numeric(
                df_work['problem_number'].astype(str).str.split('.').str[0], 
                errors='coerce'
            )
        else:
            print("ERROR: 'problem_number' column not found")
            df_work['problem_level'] = np.nan
        
        # Reshape data: create long format with agent and agent_label columns
        agents_mapping = {
            'Teacher': 'next_step_correctness_x',
            'Judge': 'next_step_correctness_y',
            'Ours': 'next_step_correctness_ours'
        }
        
        # Create list of dataframes (one per agent)
        agent_dataframes = []
        
        for agent_name, label_col in agents_mapping.items():
            if label_col in df_work.columns:
                cols_to_keep = ['count_sub', 'KG_complexity', 'problem_level', label_col]
                cols_to_keep = [c for c in cols_to_keep if c in df_work.columns]
                
                agent_df = df_work[cols_to_keep].copy()
                agent_df['agent'] = agent_name
                
                # Treat "Correct" as "Incorrect"
                agent_df['agent_label'] = agent_df[label_col].apply(
                    lambda x: 'Incorrect' if x == 'Correct' else x
                )
                agent_df = agent_df.drop(columns=[label_col])
                agent_dataframes.append(agent_df)
        
        if len(agent_dataframes) == 0:
            print("ERROR: No agent columns found")
        else:
            # Combine all agents
            df_long = pd.concat(agent_dataframes, ignore_index=True)
            
            # Remove rows with missing values
            required_cols = ['count_sub', 'agent', 'agent_label', 'KG_complexity', 'problem_level']
            print(f"\nOriginal data shape: {df_long.shape}")
            df_long = df_long.dropna(subset=required_cols)
            print(f"After removing missing values: {df_long.shape}")
            
            # ============================================================================
            # REGRESSION 1: count_sub == 3, DV = Suboptimal (1) vs Not Suboptimal (0)
            # ============================================================================
            print("\n" + "="*80)
            print("REGRESSION 1: count_sub == 3")
            print("DV: 1 if agent_label == 'Suboptimal', else 0")
            print("IVs: KG_complexity + problem_level")
            print("="*80)
            
            # Subset: count_sub == 3
            reg1_data = df_long[df_long['count_sub'] == 3].copy()
            reg1_data = reg1_data.dropna(subset=['KG_complexity', 'problem_level', 'agent_label'])
            
            if len(reg1_data) < 10:
                print(f"ERROR: Insufficient data for regression (n={len(reg1_data)})")
            else:
                # Create binary DV: 1 if Suboptimal, 0 otherwise
                reg1_data['dv'] = (reg1_data['agent_label'] == 'Suboptimal').astype(int)
                
                print(f"\nTotal observations: {len(reg1_data)}")
                print(f"Suboptimal (DV=1): {reg1_data['dv'].sum()} ({reg1_data['dv'].mean()*100:.1f}%)")
                print(f"Not Suboptimal (DV=0): {(reg1_data['dv']==0).sum()} ({(reg1_data['dv']==0).mean()*100:.1f}%)")
                
                # Prepare features
                X1 = reg1_data[['KG_complexity', 'problem_level']].copy()
                X1 = X1.astype(float)
                X1 = sm.add_constant(X1)  # Add constant for binary logistic regression
                
                y1 = reg1_data['dv'].astype(int)
                
                try:
                    # Fit binary logistic regression
                    model1 = sm.Logit(y1, X1)
                    result1 = model1.fit(disp=0, method='bfgs')
                    
                    print("\nRegression Summary:")
                    print(result1.summary())
                    
                    # Extract coefficients and odds ratios
                    print("\n" + "-"*80)
                    print("COEFFICIENTS AND ODDS RATIOS")
                    print("-"*80)
                    
                    coef_df1 = pd.DataFrame({
                        'Variable': result1.params.index,
                        'Coefficient': result1.params.values,
                        'Std_Error': result1.bse.values,
                        'P_Value': result1.pvalues.values,
                        'Odds_Ratio': np.exp(result1.params.values)
                    })
                    
                    print(coef_df1.to_string(index=False))
                    
                    # Interpretation
                    print("\n" + "-"*80)
                    print("INTERPRETATION")
                    print("-"*80)
                    for var in ['KG_complexity', 'problem_level']:
                        if var in result1.params.index:
                            coef = result1.params[var]
                            pval = result1.pvalues[var]
                            or_val = np.exp(coef)
                            sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
                            print(f"{var}: OR={or_val:.4f}, p={pval:.4f} {sig}")
                            if pval < 0.05:
                                print(f"  → For each unit increase in {var}, odds of labeling 'Suboptimal' change by {(or_val-1)*100:.1f}%")
                    
                except Exception as e:
                    print(f"Error fitting model 1: {e}")
                    import traceback
                    traceback.print_exc()
            
            # ============================================================================
            # REGRESSION 2: count_sub in {0,1,2}, DV = Incorrect (1) vs Not Incorrect (0)
            # ============================================================================
            print("\n" + "="*80)
            print("REGRESSION 2: count_sub in {0,1,2}")
            print("DV: 1 if agent_label == 'Incorrect', else 0")
            print("IVs: KG_complexity + problem_level")
            print("="*80)
            
            # Subset: count_sub in {0,1,2}
            reg2_data = df_long[df_long['count_sub'].isin([0, 1, 2])].copy()
            reg2_data = reg2_data.dropna(subset=['KG_complexity', 'problem_level', 'agent_label'])
            
            if len(reg2_data) < 10:
                print(f"ERROR: Insufficient data for regression (n={len(reg2_data)})")
            else:
                # Create binary DV: 1 if Incorrect, 0 otherwise
                reg2_data['dv'] = (reg2_data['agent_label'] == 'Incorrect').astype(int)
                
                print(f"\nTotal observations: {len(reg2_data)}")
                print(f"Incorrect (DV=1): {reg2_data['dv'].sum()} ({reg2_data['dv'].mean()*100:.1f}%)")
                print(f"Not Incorrect (DV=0): {(reg2_data['dv']==0).sum()} ({(reg2_data['dv']==0).mean()*100:.1f}%)")
                
                # Prepare features
                X2 = reg2_data[['KG_complexity', 'problem_level']].copy()
                X2 = X2.astype(float)
                X2 = sm.add_constant(X2)  # Add constant for binary logistic regression
                
                y2 = reg2_data['dv'].astype(int)
                
                try:
                    # Fit binary logistic regression
                    model2 = sm.Logit(y2, X2)
                    result2 = model2.fit(disp=0, method='bfgs')
                    
                    print("\nRegression Summary:")
                    print(result2.summary())
                    
                    # Extract coefficients and odds ratios
                    print("\n" + "-"*80)
                    print("COEFFICIENTS AND ODDS RATIOS")
                    print("-"*80)
                    
                    coef_df2 = pd.DataFrame({
                        'Variable': result2.params.index,
                        'Coefficient': result2.params.values,
                        'Std_Error': result2.bse.values,
                        'P_Value': result2.pvalues.values,
                        'Odds_Ratio': np.exp(result2.params.values)
                    })
                    
                    print(coef_df2.to_string(index=False))
                    
                    # Interpretation
                    print("\n" + "-"*80)
                    print("INTERPRETATION")
                    print("-"*80)
                    for var in ['KG_complexity', 'problem_level']:
                        if var in result2.params.index:
                            coef = result2.params[var]
                            pval = result2.pvalues[var]
                            or_val = np.exp(coef)
                            sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
                            print(f"{var}: OR={or_val:.4f}, p={pval:.4f} {sig}")
                            if pval < 0.05:
                                print(f"  → For each unit increase in {var}, odds of labeling 'Incorrect' change by {(or_val-1)*100:.1f}%")
                    
                except Exception as e:
                    print(f"Error fitting model 2: {e}")
                    import traceback
                    traceback.print_exc()
            
            # ============================================================================
            # SUMMARY TABLE
            # ============================================================================
            print("\n" + "="*80)
            print("SUMMARY TABLE")
            print("="*80)
            
            summary_data = []
            
            # Model 1 summary
            if 'result1' in locals():
                for var in ['KG_complexity', 'problem_level']:
                    if var in result1.params.index:
                        summary_data.append({
                            'Model': 'count_sub==3 (Suboptimal)',
                            'Variable': var,
                            'Coefficient': result1.params[var],
                            'Odds_Ratio': np.exp(result1.params[var]),
                            'P_Value': result1.pvalues[var]
                        })
            
            # Model 2 summary
            if 'result2' in locals():
                for var in ['KG_complexity', 'problem_level']:
                    if var in result2.params.index:
                        summary_data.append({
                            'Model': 'count_sub in {0,1,2} (Incorrect)',
                            'Variable': var,
                            'Coefficient': result2.params[var],
                            'Odds_Ratio': np.exp(result2.params[var]),
                            'P_Value': result2.pvalues[var]
                        })
            
            if summary_data:
                summary_df = pd.DataFrame(summary_data)
                summary_df = summary_df.round(4)
                print(summary_df.to_string(index=False))
            
            print("\n" + "="*80)
            print("ANALYSIS COMPLETE")
            print("="*80)

BINARY LOGISTIC REGRESSIONS
Note: 'Correct' labels are treated as 'Incorrect'

1. DATA PREPARATION
--------------------------------------------------------------------------------
Using df_non_optimal with count_sub column

Original data shape: (1344, 5)
After removing missing values: (1344, 5)

REGRESSION 1: count_sub == 3
DV: 1 if agent_label == 'Suboptimal', else 0
IVs: KG_complexity + problem_level

Total observations: 123
Suboptimal (DV=1): 11 (8.9%)
Not Suboptimal (DV=0): 112 (91.1%)

Regression Summary:
                           Logit Regression Results                           
Dep. Variable:                     dv   No. Observations:                  123
Model:                          Logit   Df Residuals:                      120
Method:                           MLE   Df Model:                            2
Date:                Tue, 27 Jan 2026   Pseudo R-squ.:                 0.04949
Time:                        22:28:34   Log-Likelihood:                -35.216
converged:

In [45]:
import pandas as pd
import numpy as np

print("="*80)
print("PATTERN ANALYSIS: count_sub VALUES vs AGENT LABELS")
print("="*80)
print("Analyzing what count_sub patterns (0, 1, 2, 3) lead to 'Suboptimal' vs 'Incorrect' labels")
print("Treating 'Correct' as 'Incorrect'")
print("="*80)

agents = {
    'Teacher': 'next_step_correctness_x',
    'Judge': 'next_step_correctness_y',
    'Ours': 'next_step_correctness_ours'
}

# ============================================================================
# PER-AGENT ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("1. PER-AGENT CROSSTAB ANALYSIS")
print("="*80)

agent_results = []

for agent_name, pred_col in agents.items():
    if pred_col not in df_non_optimal.columns:
        continue
    
    print(f"\n{agent_name}:")
    print("-" * 80)
    
    # Create binary labels (treating Correct as Incorrect)
    df_agent = df_non_optimal.copy()
    df_agent['agent_label'] = df_agent[pred_col].apply(
        lambda x: 'Incorrect' if x == 'Correct' else x
    )
    
    # Create crosstab
    crosstab = pd.crosstab(
        df_agent['count_sub'], 
        df_agent['agent_label'], 
        margins=True,
        normalize='index'  # Row percentages
    )
    
    print("\nCrosstab (Counts):")
    print(pd.crosstab(df_agent['count_sub'], df_agent['agent_label'], margins=True))
    
    print("\nCrosstab (Row Percentages):")
    print(crosstab.round(3) * 100)
    
    # Store for summary
    for count_val in [0, 1, 2, 3]:
        subset = df_agent[df_agent['count_sub'] == count_val]
        if len(subset) > 0:
            p_suboptimal = (subset['agent_label'] == 'Suboptimal').sum() / len(subset) * 100
            p_incorrect = (subset['agent_label'] == 'Incorrect').sum() / len(subset) * 100
            
            agent_results.append({
                'Agent': agent_name,
                'count_sub': count_val,
                'P(Suboptimal)': round(p_suboptimal, 2),
                'P(Incorrect)': round(p_incorrect, 2),
                'n': len(subset)
            })

# ============================================================================
# POOLED ANALYSIS ACROSS ALL AGENTS
# ============================================================================
print("\n" + "="*80)
print("2. POOLED ANALYSIS ACROSS ALL AGENTS")
print("="*80)

# Collect all data
all_data = []

for agent_name, pred_col in agents.items():
    if pred_col not in df_non_optimal.columns:
        continue
    
    df_agent = df_non_optimal.copy()
    df_agent['agent_label'] = df_agent[pred_col].apply(
        lambda x: 'Incorrect' if x == 'Correct' else x
    )
    df_agent['agent'] = agent_name
    
    all_data.append(df_agent[['count_sub', 'agent_label', 'agent']])

# Combine
combined = pd.concat(all_data, ignore_index=True)

print("\nPooled Crosstab (Counts):")
pooled_counts = pd.crosstab(combined['count_sub'], combined['agent_label'], margins=True)
print(pooled_counts)

print("\nPooled Crosstab (Row Percentages):")
pooled_pct = pd.crosstab(combined['count_sub'], combined['agent_label'], normalize='index', margins=True)
print((pooled_pct * 100).round(2))

# ============================================================================
# PATTERN SUMMARY
# ============================================================================
print("\n" + "="*80)
print("3. PATTERN SUMMARY")
print("="*80)

print("\nFor each count_sub value, what percentage of labels are Suboptimal vs Incorrect:")
print("-" * 80)

pooled_summary = []

for count_val in [0, 1, 2, 3]:
    subset = combined[combined['count_sub'] == count_val]
    if len(subset) > 0:
        p_suboptimal = (subset['agent_label'] == 'Suboptimal').sum() / len(subset) * 100
        p_incorrect = (subset['agent_label'] == 'Incorrect').sum() / len(subset) * 100
        
        # Determine dominant pattern
        if p_suboptimal > p_incorrect:
            pattern = f"→ More likely to be labeled 'Suboptimal' ({p_suboptimal:.1f}% vs {p_incorrect:.1f}%)"
        elif p_incorrect > p_suboptimal:
            pattern = f"→ More likely to be labeled 'Incorrect' ({p_incorrect:.1f}% vs {p_suboptimal:.1f}%)"
        else:
            pattern = f"→ Equal probability ({p_suboptimal:.1f}% each)"
        
        print(f"count_sub = {count_val}: {pattern} (n={len(subset)} labels)")
        
        pooled_summary.append({
            'count_sub': count_val,
            'P(Suboptimal)': round(p_suboptimal, 2),
            'P(Incorrect)': round(p_incorrect, 2),
            'n_labels': len(subset),
            'Dominant_Label': 'Suboptimal' if p_suboptimal > p_incorrect else 'Incorrect'
        })

# ============================================================================
# DETAILED PATTERN ANALYSIS
# ============================================================================
print("\n" + "="*80)
print("4. DETAILED PATTERN ANALYSIS BY AGENT")
print("="*80)

agent_summary_df = pd.DataFrame(agent_results)

if len(agent_summary_df) > 0:
    print("\nSummary Table:")
    print(agent_summary_df.to_string(index=False))
    
    # Pivot for easier comparison
    print("\nPivot Table - P(Suboptimal) by count_sub and Agent:")
    pivot_suboptimal = agent_summary_df.pivot(index='count_sub', columns='Agent', values='P(Suboptimal)')
    print(pivot_suboptimal.round(2))
    
    print("\nPivot Table - P(Incorrect) by count_sub and Agent:")
    pivot_incorrect = agent_summary_df.pivot(index='count_sub', columns='Agent', values='P(Incorrect)')
    print(pivot_incorrect.round(2))

# ============================================================================
# PATTERN TRENDS
# ============================================================================
print("\n" + "="*80)
print("5. PATTERN TRENDS")
print("="*80)

print("\nHow does labeling change as count_sub increases?")
print("-" * 80)

pooled_summary_df = pd.DataFrame(pooled_summary)

if len(pooled_summary_df) > 0:
    print("\nPooled Summary:")
    print(pooled_summary_df.to_string(index=False))
    
    # Analyze trend
    print("\nTrend Analysis:")
    for i in range(len(pooled_summary_df) - 1):
        curr = pooled_summary_df.iloc[i]
        next_val = pooled_summary_df.iloc[i + 1]
        
        curr_sub = curr['P(Suboptimal)']
        next_sub = next_val['P(Suboptimal)']
        
        change = next_sub - curr_sub
        direction = "increases" if change > 0 else "decreases" if change < 0 else "stays same"
        
        print(f"  count_sub {curr['count_sub']} → {next_val['count_sub']}: "
              f"P(Suboptimal) {direction} by {abs(change):.2f}% "
              f"({curr_sub:.1f}% → {next_sub:.1f}%)")

# ============================================================================
# KEY INSIGHTS
# ============================================================================
print("\n" + "="*80)
print("6. KEY INSIGHTS")
print("="*80)

if len(pooled_summary_df) > 0:
    # Find which count_sub values are most associated with each label
    max_suboptimal = pooled_summary_df.loc[pooled_summary_df['P(Suboptimal)'].idxmax()]
    max_incorrect = pooled_summary_df.loc[pooled_summary_df['P(Incorrect)'].idxmax()]
    
    print(f"\n• count_sub = {int(max_suboptimal['count_sub'])} has the highest P(Suboptimal): "
          f"{max_suboptimal['P(Suboptimal)']:.1f}%")
    
    print(f"• count_sub = {int(max_incorrect['count_sub'])} has the highest P(Incorrect): "
          f"{max_incorrect['P(Incorrect)']:.1f}%")
    
    # Check if there's a clear threshold
    print("\n• Labeling patterns by count_sub:")
    for _, row in pooled_summary_df.iterrows():
        if row['P(Suboptimal)'] > 50:
            print(f"  - count_sub={int(row['count_sub'])}: Agents tend to label as 'Suboptimal' "
                  f"({row['P(Suboptimal)']:.1f}%)")
        elif row['P(Incorrect)'] > 50:
            print(f"  - count_sub={int(row['count_sub'])}: Agents tend to label as 'Incorrect' "
                  f"({row['P(Incorrect)']:.1f}%)")
        else:
            print(f"  - count_sub={int(row['count_sub'])}: Mixed labeling "
                  f"(Suboptimal: {row['P(Suboptimal)']:.1f}%, Incorrect: {row['P(Incorrect)']:.1f}%)")

print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

PATTERN ANALYSIS: count_sub VALUES vs AGENT LABELS
Analyzing what count_sub patterns (0, 1, 2, 3) lead to 'Suboptimal' vs 'Incorrect' labels
Treating 'Correct' as 'Incorrect'

1. PER-AGENT CROSSTAB ANALYSIS

Teacher:
--------------------------------------------------------------------------------

Crosstab (Counts):
agent_label  Incorrect  Suboptimal  All
count_sub                              
0                  160           4  164
1                  130           1  131
2                  109           3  112
3                   40           1   41
All                439           9  448

Crosstab (Row Percentages):
agent_label  Incorrect  Suboptimal
count_sub                         
0                 97.6         2.4
1                 99.2         0.8
2                 97.3         2.7
3                 97.6         2.4
All               98.0         2.0

Judge:
--------------------------------------------------------------------------------

Crosstab (Counts):
agent_label  Incorr

# Similarity 

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

model = SentenceTransformer('all-mpnet-base-v2')

# Shared embeddings (same for all comparisons)
kg_emb = model.encode(df_non_optimal['KG_plan'].tolist())
student_emb = model.encode(df_non_optimal['student_reasoning'].tolist())

# Separate embeddings per agent
tutor_fb_emb = model.encode(df_non_optimal['teacher_feedback'].tolist())
teacher_fb_emb = model.encode(df_non_optimal['judge_feedback'].tolist())
verifier_fb_emb = model.encode(df_non_optimal['final_feedback_ours'].tolist())

In [ ]:
def compute_similarities(feedback_emb, kg_emb, student_emb):
    sim_to_kg = [cosine_similarity([f], [k])[0][0] 
                 for f, k in zip(feedback_emb, kg_emb)]
    sim_to_student = [cosine_similarity([f], [s])[0][0] 
                      for f, s in zip(feedback_emb, student_emb)]
    return sim_to_kg, sim_to_student

# Tutor
df_non_optimal['tutor_sim_kg'], df_non_optimal['tutor_sim_student'] = compute_similarities(
    tutor_fb_emb, kg_emb, student_emb)
df_non_optimal['tutor_anchor_ratio'] = df_non_optimal['tutor_sim_kg'] / df_non_optimal['tutor_sim_student']

# Teacher
df_non_optimal['teacher_sim_kg'], df_non_optimal['teacher_sim_student'] = compute_similarities(
    teacher_fb_emb, kg_emb, student_emb)
df_non_optimal['teacher_anchor_ratio'] = df_non_optimal['teacher_sim_kg'] / df_non_optimal['teacher_sim_student']

# Verifier
df_non_optimal['verifier_sim_kg'], df_non_optimal['verifier_sim_student'] = compute_similarities(
    verifier_fb_emb, kg_emb, student_emb)
df_non_optimal['verifier_anchor_ratio'] = df_non_optimal['verifier_sim_kg'] / df_non_optimal['verifier_sim_student']

In [ ]:
"""
Simplified PCA Visualization - Borchers & Shou Style
=====================================================
Creates PCA plots with covariance ellipses for LLM tutoring feedback
"""

import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import matplotlib.transforms as transforms

# ============================================================================
# HELPER: Draw Covariance Ellipse
# ============================================================================

def draw_ellipse(x, y, ax, color, label, n_std=2.0):
    """Draw 95% confidence ellipse (Borchers & Shou style)"""
    if len(x) < 3:
        return
    
    cov = np.cov(x, y)
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    
    ell_radius_x = np.sqrt(1 + pearson)
    ell_radius_y = np.sqrt(1 - pearson)
    
    ellipse = Ellipse((0, 0), width=ell_radius_x * 2, height=ell_radius_y * 2,
                      facecolor='none', edgecolor=color, linestyle='--', linewidth=2)
    
    scale_x = np.sqrt(cov[0, 0]) * n_std
    scale_y = np.sqrt(cov[1, 1]) * n_std
    
    transf = transforms.Affine2D() \
        .rotate_deg(45) \
        .scale(scale_x, scale_y) \
        .translate(np.mean(x), np.mean(y))
    
    ellipse.set_transform(transf + ax.transData)
    ax.add_patch(ellipse)

# ============================================================================
# MAIN VISUALIZATION FUNCTIONS
# ============================================================================

def create_pca_by_agent(df, model_name='all-mpnet-base-v2'):
    """
    PCA visualization colored by AGENT type (Teacher/Judge/Ours)
    
    Parameters:
    -----------
    df : DataFrame with columns:
        - teacher_feedback
        - judge_feedback  
        - final_feedback_ours
    """
    
    # Load model and compute embeddings
    model = SentenceTransformer(model_name)
    
    teacher_emb = model.encode(df['teacher_feedback'].fillna("").astype(str).tolist())
    judge_emb = model.encode(df['judge_feedback'].fillna("").astype(str).tolist())
    ours_emb = model.encode(df['final_feedback_ours'].fillna("").astype(str).tolist())
    
    # Stack all embeddings
    all_emb = np.vstack([teacher_emb, judge_emb, ours_emb])
    n = len(df)
    labels = ['Teacher']*n + ['Judge']*n + ['Ours']*n
    
    # PCA
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(all_emb)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))
    
    colors = {'Teacher': '#1f77b4', 'Judge': '#ff7f0e', 'Ours': '#2ca02c'}
    
    for agent in ['Teacher', 'Judge', 'Ours']:
        mask = np.array(labels) == agent
        x, y = reduced[mask, 0], reduced[mask, 1]
        
        ax.scatter(x, y, c=colors[agent], label=agent, alpha=0.5, s=30)
        draw_ellipse(x, y, ax, colors[agent], agent)
    
    ax.set_xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('PCA of Feedback Embeddings by Agent Type')
    ax.legend(loc='upper right')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pca_by_agent.png', dpi=300)
    plt.show()
    
    return pca, reduced


def create_pca_by_llm(df, feedback_col='teacher_feedback', llm_col='llm_model', 
                      model_name='all-mpnet-base-v2'):
    """
    PCA visualization colored by LLM model (like Borchers & Shou Figure 3)
    
    Parameters:
    -----------
    df : DataFrame with columns:
        - feedback_col: column containing feedback text
        - llm_col: column containing LLM model name
    """
    
    # Load model and compute embeddings
    model = SentenceTransformer(model_name)
    embeddings = model.encode(df[feedback_col].fillna("").astype(str).tolist())
    
    # PCA
    pca = PCA(n_components=2)
    reduced = pca.fit_transform(embeddings)
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 10))
    
    unique_llms = df[llm_col].unique()
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_llms)))
    color_map = dict(zip(unique_llms, colors))
    
    for llm in unique_llms:
        mask = df[llm_col] == llm
        x, y = reduced[mask, 0], reduced[mask, 1]
        
        ax.scatter(x, y, c=[color_map[llm]], label=llm, alpha=0.5, s=30)
        draw_ellipse(x, y, ax, color_map[llm], llm)
    
    ax.set_xlabel(f'Principal Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
    ax.set_ylabel(f'Principal Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
    ax.set_title('PCA of Feedback Embeddings by Backbone LLM')
    ax.legend(loc='upper right')
    ax.axhline(y=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.axvline(x=0, color='gray', linestyle='-', linewidth=0.5, alpha=0.5)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('pca_by_llm.png', dpi=300)
    plt.show()
    
    return pca, reduced


def create_similarity_analysis(df, model_name='all-mpnet-base-v2'):
    """
    Compute and visualize similarity between feedback, student reasoning, and KG plan
    
    Parameters:
    -----------
    df : DataFrame with columns:
        - student_reasoning (or similar)
        - KG_plan (or kg_plan)
        - teacher_feedback
        - judge_feedback
        - final_feedback_ours
    """
    
    # Load model
    model = SentenceTransformer(model_name)
    
    # Compute embeddings
    print("Computing embeddings...")
    student_emb = model.encode(df['student_reasoning'].fillna("").astype(str).tolist())
    kg_emb = model.encode(df['KG_plan'].fillna("").astype(str).tolist())
    teacher_emb = model.encode(df['teacher_feedback'].fillna("").astype(str).tolist())
    judge_emb = model.encode(df['judge_feedback'].fillna("").astype(str).tolist())
    ours_emb = model.encode(df['final_feedback_ours'].fillna("").astype(str).tolist())
    
    # Compute pairwise similarities
    def pairwise_sim(emb1, emb2):
        return [cosine_similarity([e1], [e2])[0][0] for e1, e2 in zip(emb1, emb2)]
    
    results = {
        'teacher_vs_student': pairwise_sim(teacher_emb, student_emb),
        'teacher_vs_kg': pairwise_sim(teacher_emb, kg_emb),
        'judge_vs_student': pairwise_sim(judge_emb, student_emb),
        'judge_vs_kg': pairwise_sim(judge_emb, kg_emb),
        'ours_vs_student': pairwise_sim(ours_emb, student_emb),
        'ours_vs_kg': pairwise_sim(ours_emb, kg_emb),
        'teacher_vs_judge': pairwise_sim(teacher_emb, judge_emb),
        'teacher_vs_ours': pairwise_sim(teacher_emb, ours_emb),
        'judge_vs_ours': pairwise_sim(judge_emb, ours_emb),
    }
    
    # -------------------------
    # PLOT 1: Anchoring Scatter
    # -------------------------
    fig, ax = plt.subplots(figsize=(10, 10))
    
    colors = {'Teacher': '#1f77b4', 'Judge': '#ff7f0e', 'Ours': '#2ca02c'}
    
    ax.scatter(results['teacher_vs_student'], results['teacher_vs_kg'],
               c=colors['Teacher'], label='Teacher', alpha=0.5, s=30)
    ax.scatter(results['judge_vs_student'], results['judge_vs_kg'],
               c=colors['Judge'], label='Judge', alpha=0.5, s=30)
    ax.scatter(results['ours_vs_student'], results['ours_vs_kg'],
               c=colors['Ours'], label='Ours', alpha=0.5, s=30)
    
    ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Balanced (ratio=1)')
    
    ax.set_xlabel('Similarity to Student Reasoning')
    ax.set_ylabel('Similarity to KG Plan')
    ax.set_title('Solution Anchoring: Feedback Similarity')
    ax.legend()
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('anchoring_scatter.png', dpi=300)
    plt.show()
    
    # -------------------------
    # PLOT 2: Anchoring Boxplot
    # -------------------------
    import seaborn as sns
    
    ratios = {
        'Teacher': np.array(results['teacher_vs_kg']) / (np.array(results['teacher_vs_student']) + 1e-8),
        'Judge': np.array(results['judge_vs_kg']) / (np.array(results['judge_vs_student']) + 1e-8),
        'Ours': np.array(results['ours_vs_kg']) / (np.array(results['ours_vs_student']) + 1e-8),
    }
    
    plot_data = []
    for agent, vals in ratios.items():
        for v in vals:
            plot_data.append({'Agent': agent, 'Anchoring Ratio': v})
    
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.boxplot(x='Agent', y='Anchoring Ratio', data=pd.DataFrame(plot_data), ax=ax,
                palette=colors)
    ax.axhline(y=1, color='red', linestyle='--', label='Balanced')
    ax.set_title('Solution Anchoring Ratio by Agent')
    ax.legend()
    
    plt.tight_layout()
    plt.savefig('anchoring_boxplot.png', dpi=300)
    plt.show()
    
    # -------------------------
    # Print Statistics
    # -------------------------
    print("\n" + "="*60)
    print("SIMILARITY STATISTICS")
    print("="*60)
    
    print("\n--- Feedback vs Student ---")
    for agent in ['teacher', 'judge', 'ours']:
        vals = results[f'{agent}_vs_student']
        print(f"  {agent.capitalize():10s}: Mean={np.mean(vals):.3f}, SD={np.std(vals):.3f}")
    
    print("\n--- Feedback vs KG ---")
    for agent in ['teacher', 'judge', 'ours']:
        vals = results[f'{agent}_vs_kg']
        print(f"  {agent.capitalize():10s}: Mean={np.mean(vals):.3f}, SD={np.std(vals):.3f}")
    
    print("\n--- Anchoring Ratio ---")
    for agent in ['Teacher', 'Judge', 'Ours']:
        vals = ratios[agent]
        print(f"  {agent:10s}: Mean={np.mean(vals):.3f}, >1: {np.mean(vals > 1)*100:.1f}%")
    
    print("\n--- Inter-Agent Similarity ---")
    print(f"  Teacher-Judge:    {np.mean(results['teacher_vs_judge']):.3f}")
    print(f"  Teacher-Ours:     {np.mean(results['teacher_vs_ours']):.3f}")
    print(f"  Judge-Ours:       {np.mean(results['judge_vs_ours']):.3f}")
    
    return results, ratios


# ============================================================================
# QUICK USAGE
# ============================================================================

# In your notebook cell:
create_pca_by_agent(df_non_optimal)

# If you want similarity analysis, make sure you have the right column names:
# Check what columns you have:
print([c for c in df_non_optimal.columns if 'student' in c.lower() or 'reasoning' in c.lower()])
print([c for c in df_non_optimal.columns if 'kg' in c.lower() or 'plan' in c.lower()])

# Then call:
create_similarity_analysis(df_non_optimal)

# Rule normalization 

In [ ]:
# Robust normalization function for rules
def normalize_rule(rule):
    """
    Normalize a rule by handling None/NaN, removing spaces, and converting to short name.
    Returns empty string for invalid rules to ensure consistent comparison.
    """
    if rule is None or pd.isna(rule):
        return ''
    
    rule_str = str(rule).strip()
    if not rule_str or rule_str.lower() in ['nan', 'none', '']:
        return ''
    
    # Remove spaces and convert to short name
    rule_str = rule_str.replace(' ', '')
    normalized = convert_rule_to_short_name(rule_str)
    
    # Ensure we return a string, not None
    return str(normalized) if normalized else ''

# Normalize all three rules consistently for robust comparison
df['teacher_rule_normalized'] = df['teacher_rule'].apply(normalize_rule)
df['student_rule_normalized'] = df['student_rule'].apply(normalize_rule)
df['KG_rule_normalized'] = df['KG_rule'].apply(normalize_rule)

# Filter rows where teacher_rule == student_rule and student_rule != KG_rule
# Using normalized versions for robust comparison
df_filtered = df[
    (df['teacher_rule_normalized'] == df['student_rule_normalized']) & 
    (df['student_rule_normalized'] != df['KG_rule_normalized']) &
    (df['student_rule_normalized'] != '') &  # Exclude empty/invalid rules
    (df['teacher_rule_normalized'] != '')  # Exclude empty/invalid rules
].copy()

# Validation: Check for any normalization issues
print("=" * 80)
print("RULE NORMALIZATION VALIDATION")
print("=" * 80)
print(f"\nTotal rows: {len(df)}")
print(f"\nRule normalization summary:")
print(f"  Teacher rules normalized: {df['teacher_rule_normalized'].notna().sum()}")
print(f"  Student rules normalized: {df['student_rule_normalized'].notna().sum()}")
print(f"  KG rules normalized: {df['KG_rule_normalized'].notna().sum()}")
print(f"\nEmpty/invalid rules (after normalization):")
print(f"  Teacher rules empty: {(df['teacher_rule_normalized'] == '').sum()}")
print(f"  Student rules empty: {(df['student_rule_normalized'] == '').sum()}")
print(f"  KG rules empty: {(df['KG_rule_normalized'] == '').sum()}")

print("\n" + "=" * 80)
print("ROWS WHERE student_rule == teacher_rule AND student_rule != KG_rule")
print("=" * 80)
print(f"Count: {len(df_filtered)}")


# Teacher rules vs KG rules 

In [ ]:
# Robust computation of teacher rule matches with KG rules
print("=" * 80)
print("TEACHER RULE vs KG RULE MATCH ANALYSIS")
print("=" * 80)

# Ensure we have normalized rules (reuse from previous cell if available, otherwise normalize)
if 'teacher_rule_normalized' not in df.columns or 'KG_rule_normalized' not in df.columns:
    # Reuse the normalize_rule function from Cell 12 if available, otherwise define it
    if 'normalize_rule' not in globals():
        def normalize_rule(rule):
            """Normalize a rule by handling None/NaN, removing spaces, and converting to short name."""
            if rule is None or pd.isna(rule):
                return ''
            rule_str = str(rule).strip()
            if not rule_str or rule_str.lower() in ['nan', 'none', '']:
                return ''
            rule_str = rule_str.replace(' ', '')
            normalized = convert_rule_to_short_name(rule_str)
            return str(normalized) if normalized else ''
    
    df['teacher_rule_normalized'] = df['teacher_rule'].apply(normalize_rule)
    df['KG_rule_normalized'] = df['KG_rule'].apply(normalize_rule)

# Compute matches
df['teacher_kg_rule_match'] = (df['teacher_rule_normalized'] == df['KG_rule_normalized']).astype(int)

# Filter out rows with empty/invalid rules for accurate statistics
valid_mask = (df['teacher_rule_normalized'] != '') & (df['KG_rule_normalized'] != '')
df_valid = df[valid_mask].copy()

# Overall statistics
total_rows = len(df)
valid_rows = len(df_valid)
matches = df_valid['teacher_kg_rule_match'].sum()
non_matches = valid_rows - matches

print(f"\n📊 OVERALL STATISTICS")
print(f"{'='*80}")
print(f"Total rows: {total_rows}")
print(f"Valid rows (both rules non-empty): {valid_rows} ({valid_rows/total_rows*100:.1f}%)")
print(f"Invalid rows (empty rules): {total_rows - valid_rows} ({(total_rows - valid_rows)/total_rows*100:.1f}%)")
print(f"\n✅ Teacher rule matches KG rule: {matches} ({matches/valid_rows*100:.1f}%)")
print(f"❌ Teacher rule does NOT match KG rule: {non_matches} ({non_matches/valid_rows*100:.1f}%)")

# Detailed breakdown
print(f"\n📈 DETAILED BREAKDOWN")
print(f"{'='*80}")

# Show matches vs non-matches
print(f"\nMatch Status:")
match_df = df_valid['teacher_kg_rule_match'].value_counts().sort_index()
for status, count in match_df.items():
    status_str = "MATCH" if status == 1 else "NO MATCH"
    print(f"  {status_str}: {count} ({count/valid_rows*100:.1f}%)")

# Show rule distribution for matches
print(f"\n📋 RULE DISTRIBUTION (Matches)")
print(f"{'='*80}")
matches_df = df_valid[df_valid['teacher_kg_rule_match'] == 1]
if len(matches_df) > 0:
    print(f"\nMost common matching rules:")
    matching_rules = matches_df['teacher_rule_normalized'].value_counts().head(10)
    for rule, count in matching_rules.items():
        print(f"  {rule}: {count} ({count/len(matches_df)*100:.1f}%)")


non_matches_df = df_valid[df_valid['teacher_kg_rule_match'] == 0]
if len(non_matches_df) > 0:
    
    teacher_rules_nonmatch = non_matches_df['teacher_rule_normalized'].value_counts().head(10)
    print(f"\n🔴 TOP TEACHER RULES (non-matching):")
    print("   (What rules the teacher predicted when they disagreed with KG)")
    print("   Count = How many times teacher used this rule incorrectly")
    for rule, count in teacher_rules_nonmatch.items():
        pct = count / len(non_matches_df) * 100
        print(f"  {rule}: {count} times ({pct:.1f}% of non-matches)")
    
    print(f"\n🟢 TOP KG RULES (non-matching):")
    print("   (What rules the KG had when teacher disagreed)")
    print("   Count = How many times KG had this correct rule that teacher missed")
    kg_rules_nonmatch = non_matches_df['KG_rule_normalized'].value_counts().head(10)
    for rule, count in kg_rules_nonmatch.items():
        pct = count / len(non_matches_df) * 100
        print(f"  {rule}: {count} times ({pct:.1f}% of non-matches)")
    
    # Show side-by-side comparison for better understanding
    print(f"\n📊 SIDE-BY-SIDE COMPARISON")
    print("="*80)
    print("This shows specific mismatches - what teacher said vs what KG said:")
    
    # Create a cross-tabulation to show the most common mismatches
    mismatch_crosstab = pd.crosstab(
        non_matches_df['teacher_rule_normalized'], 
        non_matches_df['KG_rule_normalized']
    )
    
    # Get top mismatches
    print("\nMost common specific mismatches (Teacher Rule → KG Rule):")
    mismatch_pairs = []
    for teacher_rule in mismatch_crosstab.index:
        for kg_rule in mismatch_crosstab.columns:
            count = mismatch_crosstab.loc[teacher_rule, kg_rule]
            if count > 0:
                mismatch_pairs.append((teacher_rule, kg_rule, count))
    
    # Sort by count and show top 10
    mismatch_pairs.sort(key=lambda x: x[2], reverse=True)
    for teacher_rule, kg_rule, count in mismatch_pairs[:10]:
        print(f"  Teacher: {teacher_rule} → KG: {kg_rule} ({count} times)")


# Create summary dataframe
summary_data = {
    'Metric': [
        'Total Rows',
        'Valid Rows (both rules non-empty)',
        'Invalid Rows (empty rules)',
        'Teacher Rule Matches KG Rule',
        'Teacher Rule Does NOT Match KG Rule',
        'Match Rate (%)'
    ],
    'Count': [
        total_rows,
        valid_rows,
        total_rows - valid_rows,
        matches,
        non_matches,
        f"{matches/valid_rows*100:.2f}%" if valid_rows > 0 else "N/A"
    ],
    'Percentage': [
        '100.0%',
        f"{valid_rows/total_rows*100:.1f}%",
        f"{(total_rows - valid_rows)/total_rows*100:.1f}%",
        f"{matches/valid_rows*100:.1f}%" if valid_rows > 0 else "N/A",
        f"{non_matches/valid_rows*100:.1f}%" if valid_rows > 0 else "N/A",
        f"{matches/valid_rows*100:.2f}%" if valid_rows > 0 else "N/A"
    ]
}

summary_df = pd.DataFrame(summary_data)
print(f"\n📊 SUMMARY TABLE")
print(f"{'='*80}")
print(summary_df.to_string(index=False))

print(f"\n{'='*80}")
print("Analysis complete! The 'teacher_kg_rule_match' column has been added to the dataframe.")
print(f"{'='*80}")


# responses 

In [ ]:
# Filter: teacher_rule == student_rule, student_rule != KG_rule, count_sub != 3 
df_non_optimal['teacher_rule_normalized'] = df_non_optimal['teacher_rule'].apply(normalize_rule)
df_non_optimal['student_rule_normalized'] = df_non_optimal['student_rule'].apply(normalize_rule)
df_non_optimal['KG_rule_normalized'] = df_non_optimal['KG_rule'].apply(normalize_rule)
results = df_non_optimal[
    (df_non_optimal['teacher_rule_normalized'] == df_non_optimal['student_rule_normalized']) &
    (df_non_optimal['student_rule_normalized'] != df_non_optimal['KG_rule_normalized']) &
    (df_non_optimal['count_sub'] != 3) &
    (df_non_optimal['student_update_label'].astype(str) == 'No Match') & 
    (df_non_optimal['judge_student_update_label'].astype(str) == 'No Match') & 
    (df_non_optimal['ours_student_update_label'].astype(str) == 'No Match')].copy()
print(results.shape)
results['problem_number'].value_counts()
results[['known_expressions', 'KG_plan', 'student_reasoning', 'student_candidates']].copy().to_clipboard(index=False, excel=True)
results[['teacher_feedback', 'judge_feedback', 'final_feedback_ours']].copy().to_clipboard(index=False, excel=True)


In [ ]:
from sentence_transformers import SentenceTransformer
from rouge_score import rouge_scorer

model = SentenceTransformer("all-MiniLM-L6-v2")
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

a = df_non_optimal["judge_feedback"].fillna("").astype(str).tolist()
b = df_non_optimal["KG_plan"].fillna("").astype(str).tolist()

a_emb = model.encode(a, convert_to_tensor=True, normalize_embeddings=True)
b_emb = model.encode(b, convert_to_tensor=True, normalize_embeddings=True)

df_non_optimal["bert_score_judgefeedback_kgplan"] = (a_emb * b_emb).sum(dim=1).cpu().numpy()
df_non_optimal["rougeL_judgefeedback_kgplan"] = [scorer.score(x, y)["rougeL"].fmeasure for x, y in zip(a, b)]

df_non_optimal["bertscore_plus_rougeL"] = df_non_optimal["bert_score_judgefeedback_kgplan"] + df_non_optimal["rougeL_judgefeedback_kgplan"]




In [ ]:
results = df_non_optimal.nlargest(10, "bertscore_plus_rougeL")
results[['known_expressions', 'KG_plan', 'student_reasoning', 'student_candidates']].copy().to_clipboard(index=False, excel=True)
results[['teacher_feedback', 'judge_feedback', 'final_feedback_ours']].copy().to_clipboard(index=False, excel=True)

In [ ]:
# Filter: teacher_rule == student_rule, student_rule != KG_rule, count_sub != 3 
df_non_optimal['teacher_rule_normalized'] = df_non_optimal['teacher_rule'].apply(normalize_rule)
df_non_optimal['student_rule_normalized'] = df_non_optimal['student_rule'].apply(normalize_rule)
df_non_optimal['KG_rule_normalized'] = df_non_optimal['KG_rule'].apply(normalize_rule)
results = df_non_optimal[
    (df_non_optimal['teacher_rule_normalized'] == df_non_optimal['student_rule_normalized']) &
    (df_non_optimal['student_rule_normalized'] != df_non_optimal['KG_rule_normalized']) &
    (df_non_optimal['count_sub'] != 3) &
    (df_non_optimal['student_update_label'].astype(str) == 'No Match') & 
    (df_non_optimal['judge_student_update_label'].astype(str) == 'No Match') & 
    (df_non_optimal['ours_student_update_label'].astype(str) == 'No Match')].copy()
print(results.shape)
results['problem_number'].value_counts()
results[['known_expressions', 'KG_plan', 'student_step', 'student_rule','student_reasoning', 'student_candidates', 'teacher_feedback', 'judge_feedback', 'final_feedback_ours']].copy().to_clipboard(index=False, excel=True)
